# FineTuneHuBERTV1_Optuna_Final.ipynb

Notebook utama HuBERT+LoRA pada strict split sebagai pembanding adil terhadap WavLM.
Perlakuan dibuat sama dengan WavLM: loss MAE/L1, Optuna 50 trial, final 10 seeds, early stopping, metric selection berdasarkan validation S = 1 - MAE, serta export tabel/figure lengkap.

Notebook ini **tidak melakukan download dataset**. Jalankan sanity notebook terlebih dahulu untuk menyiapkan `data/manifest_strict_vast.csv`.


## Cara menjalankan notebook utama via tmux + papermill

Notebook utama mengasumsikan `data/manifest_strict_vast.csv` dan folder audio sudah tersedia dari sanity notebook. Jika `manifest_strict_vast.csv` belum ada tetapi `manifest_strict.csv` dan audio sudah tersedia, notebook akan mencoba membangunnya otomatis.

Dari root workspace di Vast.ai:

```bash
tmux new -s hubert_main
cd /workspace/finetunev2
mkdir -p logs
set -o pipefail
PROJECT_ROOT=/workspace/finetunev2 papermill notebooks/FineTuneHuBERTV1_Optuna_Final.ipynb notebooks/FineTuneHuBERTV1_Optuna_Final_executed.ipynb --log-output 2>&1 | tee logs/papermill_stdout.log
```

Detach tanpa mematikan proses:

```text
Ctrl+b lalu d
```

Cek progress:

```bash
tmux attach -t hubert_main
# atau, untuk log ringkas yang ditulis langsung oleh notebook:
cd /workspace/finetunev2
tail -f logs/train_live.log
# atau, untuk stdout papermill lengkap:
cd /workspace/finetunev2
tail -f logs/papermill_stdout.log
```

Catatan aman:

- Jangan klik Restart Kernel, Interrupt Kernel, atau Run All pada notebook yang sedang dieksekusi oleh papermill.
- Browser/Jupyter boleh dibuka untuk melihat file output, tetapi jangan menjalankan ulang cell pada notebook yang sedang diproses.
- File logs/train_live.log berisi log ringkas dari dalam notebook.
- File logs/papermill_stdout.log berisi output lengkap dari proses papermill.
- Jika notebook utama selesai dengan sukses, hasil executed notebook akan tersimpan sebagai: 
```text
notebooks/FineTuneHuBERTV1_Optuna_Final_executed.ipynb
```


In [1]:
# =========================
# Optional dependency check / install
# =========================
# Set AUTO_INSTALL=False if you want the notebook to fail fast when a package is missing.
AUTO_INSTALL = False

import importlib
import subprocess
import sys

# import_name -> pip_name
REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "torch": "torch",
    "soundfile": "soundfile",
    "librosa": "librosa",
    "sklearn": "scikit-learn",
    "optuna": "optuna",
    "transformers": "transformers",
    "peft": "peft",
    "IPython": "ipython",
}

missing = []
for import_name, pip_name in REQUIRED.items():
    try:
        importlib.import_module(import_name)
    except Exception:
        missing.append(pip_name)

print("Missing packages:", missing)
if missing and AUTO_INSTALL:
    cmd = [sys.executable, "-m", "pip", "install", "-U"] + missing
    print("Installing:", " ".join(cmd))
    subprocess.check_call(cmd)
elif missing:
    raise RuntimeError(f"Missing packages: {missing}")
else:
    print("All required packages are available.")


Missing packages: []
All required packages are available.


In [2]:
# =========================
# Imports and logging behavior
# =========================
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import sys
import gc
import json
import math
import time
import random
import shutil
import warnings
import zipfile
import tarfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# IMPORTANT:
# Use terminal tqdm instead of notebook widget tqdm.
# This keeps monitoring visible in tmux/papermill and avoids
# application/vnd.jupyter.widget-view+json outputs in the saved notebook.
from tqdm.std import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import soundfile as sf
import librosa

from sklearn.metrics import r2_score

import optuna
from optuna.trial import TrialState

from transformers import AutoFeatureExtractor, AutoModel, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


Python: 3.14.3 | packaged by conda-forge | (main, Feb  9 2026, 21:56:02) [GCC 14.3.0]
Torch: 2.11.0+cu130
CUDA available: True
CUDA device: NVIDIA RTX A6000


In [3]:
# =========================
# Root and folder architecture
# =========================
# If this notebook is run from the `notebooks/` folder, PROJECT_ROOT becomes its parent.
# You can override by setting env var PROJECT_ROOT=/workspace/your_project.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", str(_cwd))).resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ROOT = PROJECT_ROOT
NOTEBOOKS = ROOT / "notebooks"
DATA = ROOT / "data"

OUTPUTS = ROOT / "outputs"
CHECKPOINTS = ROOT / "checkpoints"
LOGS = ROOT / "logs"
CACHE = ROOT / "cache"

OUT_OPTUNA_STUDY = OUTPUTS / "optuna_study"
OUT_OPTUNA_TRIALS = OUTPUTS / "optuna_trials"
OUT_FINAL_RUNS = OUTPUTS / "final_runs"
OUT_FINAL_AGG = OUTPUTS / "final_aggregate"
OUT_FINAL_SELECTED = OUTPUTS / "final_selected"
OUT_APPENDIX = OUTPUTS / "appendix_ready"
OUT_TABLES = OUTPUTS / "tables"
OUT_FIGURES = OUTPUTS / "figures"

CKPT_OPTUNA_TRIALS = CHECKPOINTS / "optuna_trials"
CKPT_FINAL_RUNS = CHECKPOINTS / "final_runs"
CKPT_FINAL_SELECTED = CHECKPOINTS / "final_selected"
CKPT_SELECTED_BEST = CKPT_FINAL_SELECTED / "best_val"
CKPT_SELECTED_MEDIAN = CKPT_FINAL_SELECTED / "median"
CKPT_SELECTED_ENSEMBLE = CKPT_FINAL_SELECTED / "ensemble_sources"

HF_CACHE = CACHE / "hf_cache"
TMP_CACHE = CACHE / "tmp"

for p in [
    ROOT, NOTEBOOKS, DATA, OUTPUTS, CHECKPOINTS, LOGS, CACHE,
    OUT_OPTUNA_STUDY, OUT_OPTUNA_TRIALS, OUT_FINAL_RUNS, OUT_FINAL_AGG,
    OUT_FINAL_SELECTED, OUT_APPENDIX, OUT_TABLES, OUT_FIGURES,
    CKPT_OPTUNA_TRIALS, CKPT_FINAL_RUNS, CKPT_FINAL_SELECTED,
    CKPT_SELECTED_BEST, CKPT_SELECTED_MEDIAN, CKPT_SELECTED_ENSEMBLE,
    HF_CACHE, TMP_CACHE,
]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE)

STUDY_LOG = LOGS / "study_run.log"
TRIAL_FAIL_LOG = LOGS / "trial_failures.jsonl"
LIVE_LOG = LOGS / "train_live.log"

print("ROOT:", ROOT)
print("DATA:", DATA)
print("OUTPUTS:", OUTPUTS)
print("CHECKPOINTS:", CHECKPOINTS)
print("LOGS:", LOGS)


ROOT: /workspace/hubert_finetune
DATA: /workspace/hubert_finetune/data
OUTPUTS: /workspace/hubert_finetune/outputs
CHECKPOINTS: /workspace/hubert_finetune/checkpoints
LOGS: /workspace/hubert_finetune/logs


In [4]:
# =========================
# Global config — MAIN HuBERT+LoRA experiment
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ENABLED = (DEVICE == "cuda")

MODEL_NAME = "facebook/hubert-base-ls960"
EXPERIMENT_NAME = "HuBERT_LoRA_Strict"
SR_TARGET = 16000
MAX_SEC = 15.0
MAX_LEN = int(SR_TARGET * MAX_SEC)

# Manifest columns
SOURCE_AUDIO_COL = "audio_out"
AUDIO_COL = "audio_path"
CLIP_ID_COL = "clip_id"
GROUP_ID_COL = "group_id"
SPLIT_COL = "split_strict"

TRAIN_SPLIT_VALUE = "train"
VAL_SPLIT_VALUE = "val"
TEST_SPLIT_VALUE = "test"

LABEL_COLS = ["extraversion", "neuroticism", "agreeableness", "conscientiousness", "openness"]

# Fixed training config: kept aligned with WavLM notebook for fair comparison.
BATCH_SIZE = 40
GRAD_ACCUM_STEPS = 1
MAX_EPOCHS = 100
PATIENCE = 5
MIN_DELTA = 0.0
GRAD_CLIP = 1.0
HEAD_DROPOUT = 0.1

NUM_WORKERS = 0
PIN_MEMORY = (DEVICE == "cuda")
PERSISTENT_WORKERS = False

# Optuna: kept aligned with WavLM notebook.
N_TRIALS = 50
OPTUNA_TOPK = 5
OPTUNA_SEED = 42
SELECTION_METRIC = "S"
STUDY_NAME = "hubert_lora_strict_optuna"
STUDY_DB = OUT_OPTUNA_STUDY / "hubert_lora_optuna.db"
STUDY_STORAGE = f"sqlite:///{STUDY_DB}"

# Final runs: same seeds as WavLM.
FINAL_SEEDS = [42, 52, 62, 72, 82, 92, 102, 112, 122, 132]
N_FINAL_RUNS = len(FINAL_SEEDS)

DISPLAY_TRIAL_PLOTS = False
DISPLAY_FINAL_RUN_PLOTS = False
DISPLAY_SUMMARY_PLOTS = True

PRUNER = optuna.pruners.MedianPruner(
    n_startup_trials=10,
    n_warmup_steps=8,
    interval_steps=1,
    n_min_trials=5,
)


print("DEVICE:", DEVICE)
print("AMP_ENABLED:", AMP_ENABLED)
print("MODEL_NAME:", MODEL_NAME)
print("BATCH_SIZE:", BATCH_SIZE)
print("MAX_EPOCHS:", MAX_EPOCHS)
print("PATIENCE:", PATIENCE)
print("N_TRIALS:", N_TRIALS)
print("FINAL_SEEDS:", FINAL_SEEDS)
assert GRAD_ACCUM_STEPS == 1, "Notebook ini belum mengimplementasikan gradient accumulation; pertahankan GRAD_ACCUM_STEPS=1."
print("STUDY_STORAGE:", STUDY_STORAGE)


DEVICE: cuda
AMP_ENABLED: True
MODEL_NAME: facebook/hubert-base-ls960
BATCH_SIZE: 40
MAX_EPOCHS: 100
PATIENCE: 5
N_TRIALS: 50
FINAL_SEEDS: [42, 52, 62, 72, 82, 92, 102, 112, 122, 132]
STUDY_STORAGE: sqlite:////workspace/hubert_finetune/outputs/optuna_study/hubert_lora_optuna.db


In [5]:
# =========================
# Utilities
# =========================
def log_line(msg: str, log_path: Optional[Path] = None, also_print: bool = True):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    if also_print:
        print(line, flush=True)
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


def section(title: str):
    bar = "=" * 80
    log_line(f"\n{bar}\n{title}\n{bar}", LIVE_LOG)


def save_text(text: str, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def append_jsonl(obj: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(make_json_safe(obj), ensure_ascii=False, allow_nan=False) + "\n")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_json_safe(obj):
    """Convert numpy/pandas/path/torch objects into JSON-serializable Python objects."""

    if obj is None:
        return None

    if isinstance(obj, (str, bytes)):
        return obj.decode("utf-8", errors="replace") if isinstance(obj, bytes) else obj

    if isinstance(obj, (bool, np.bool_)):
        return bool(obj)

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, int):
        return obj

    if isinstance(obj, (np.floating, float)):
        val = float(obj)
        return val if np.isfinite(val) else None

    if isinstance(obj, torch.Tensor):
        return make_json_safe(obj.detach().cpu().tolist())

    if isinstance(obj, np.ndarray):
        return make_json_safe(obj.tolist())

    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()

    if isinstance(obj, pd.Series):
        return make_json_safe(obj.to_dict())

    if isinstance(obj, pd.DataFrame):
        return make_json_safe(obj.to_dict(orient="records"))

    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple, set)):
        return [make_json_safe(v) for v in obj]

    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass

    return obj


def save_json(obj: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(obj), f, indent=2, ensure_ascii=False, allow_nan=False)
        

def save_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def copy_if_exists(src: Path, dst: Path):
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        return True
    return False

def safe_torch_load(path: Path, map_location=None):
    """Load checkpoints robustly across PyTorch versions."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def safe_to_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def gpu_flush(reset_peak: bool = True):
    gc.collect()
    plt.close("all")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        if reset_peak:
            try:
                torch.cuda.reset_peak_memory_stats()
            except Exception:
                pass


def del_and_flush(*objs):
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    gpu_flush()


def print_gpu_memory(prefix="GPU"):
    if not torch.cuda.is_available():
        return
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    log_line(f"[{prefix}] allocated={allocated:.2f} GB | reserved={reserved:.2f} GB", LIVE_LOG)

section("Utilities loaded")


[2026-05-28 20:03:08] 
Utilities loaded


In [6]:
import transformers, peft, sklearn, librosa, optuna
env_info = {
    "python": sys.version,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "sklearn": sklearn.__version__,
    "librosa": librosa.__version__,
    "optuna": optuna.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
save_json(env_info, OUT_TABLES / "environment_info.json")

In [7]:
# =========================
# Prepare manifest_strict_vast.csv and audit strict split
# =========================
section("Cell | Prepare manifest_strict_vast.csv and audit split")

MANIFEST = DATA / "manifest_strict.csv"
MANIFEST_VAST = DATA / "manifest_strict_vast.csv"


def find_audio_dir(data_dir: Path) -> Path:
    preferred = [
        data_dir / "preprocessed_full" / "preprocessed_full",
        data_dir / "preprocessed_full",
    ]
    candidates = []
    for p in preferred:
        if p.exists() and p.is_dir():
            count = len(list(p.rglob("*.wav")))
            if count > 0:
                candidates.append((count, p))
    # Fallback: search directories containing wav files.
    for p in data_dir.rglob("*"):
        if p.is_dir():
            try:
                count = len(list(p.glob("*.wav")))
            except Exception:
                count = 0
            if count > 0:
                candidates.append((count, p))
    if not candidates:
        raise FileNotFoundError(f"No directory containing .wav files found under {data_dir}")
    candidates = sorted(set(candidates), key=lambda x: x[0], reverse=True)
    log_line("Audio dir candidates:", LIVE_LOG)
    for count, p in candidates[:10]:
        log_line(f"- {p} | wav_count={count}", LIVE_LOG)
    selected = candidates[0][1]
    expected = data_dir / "preprocessed_full" / "preprocessed_full"
    if expected.exists() and expected.is_dir():
        expected_count = len(list(expected.rglob("*.wav")))
        if expected_count > 0 and selected.resolve() != expected.resolve():
            log_line(
                f"[WARN] Auto-selected audio dir differs from WavLM expected path. "
                f"selected={selected}, expected={expected}. Using expected path for fairness.",
                LIVE_LOG,
            )
            selected = expected

    log_line(f"Final selected AUDIO_DIR: {selected}", LIVE_LOG)
    return selected


def normalize_split_values(df_: pd.DataFrame) -> pd.DataFrame:
    df_ = df_.copy()
    df_[SPLIT_COL] = df_[SPLIT_COL].astype(str).str.strip().str.lower()
    mapping = {
        "validation": "val",
        "valid": "val",
        "dev": "val",
        "train_strict": "train",
        "val_strict": "val",
        "validation_strict": "val",
        "test_strict": "test",
    }
    df_[SPLIT_COL] = df_[SPLIT_COL].replace(mapping)
    return df_


def build_manifest_vast(force_rebuild: bool = True) -> pd.DataFrame:
    assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"
    audio_dir = find_audio_dir(DATA)
    log_line(f"Selected AUDIO_DIR: {audio_dir}", LIVE_LOG)

    dfm = pd.read_csv(MANIFEST)
    assert SPLIT_COL in dfm.columns, f"Missing split column {SPLIT_COL}. Columns: {dfm.columns.tolist()}"
    assert CLIP_ID_COL in dfm.columns, f"Missing clip column {CLIP_ID_COL}"
    assert GROUP_ID_COL in dfm.columns, f"Missing group column {GROUP_ID_COL}"
    missing_labels = [c for c in LABEL_COLS if c not in dfm.columns]
    assert not missing_labels, f"Missing label columns: {missing_labels}"

    dfm = normalize_split_values(dfm)

    wav_files = list(audio_dir.rglob("*.wav"))
    wav_by_name = {p.name: p for p in wav_files}
    wav_by_stem = {p.stem: p for p in wav_files}
    log_line(f"Indexed wav files: {len(wav_files)}", LIVE_LOG)

    def resolve_audio(row):
        names = []
        if SOURCE_AUDIO_COL in row.index:
            raw = str(row[SOURCE_AUDIO_COL])
            if raw and raw.lower() != "nan":
                names.append(Path(raw).name)
                names.append(Path(raw).stem)
        clip_id = str(row[CLIP_ID_COL])
        names.extend([clip_id, f"{clip_id}.wav", Path(clip_id).stem, f"{Path(clip_id).stem}.wav"])
        for name in names:
            if name in wav_by_name:
                return str(wav_by_name[name])
            if name in wav_by_stem:
                return str(wav_by_stem[name])
        return str(audio_dir / (Path(str(row[CLIP_ID_COL])).stem + ".wav"))

    dfm[AUDIO_COL] = dfm.apply(resolve_audio, axis=1)
    exists = dfm[AUDIO_COL].apply(lambda s: Path(s).exists())
    missing_audio = dfm.loc[~exists, [CLIP_ID_COL, GROUP_ID_COL, SPLIT_COL, AUDIO_COL]].copy()
    if len(missing_audio):
        missing_csv = DATA / "manifest_missing_audio.csv"
        save_csv(missing_audio, missing_csv)
        display(missing_audio.head(10))
        raise FileNotFoundError(f"Missing audio: {len(missing_audio)}. Saved list to {missing_csv}")

    save_csv(dfm, MANIFEST_VAST)
    log_line(f"Saved manifest: {MANIFEST_VAST} | rows={len(dfm)}", LIVE_LOG)
    return dfm


def audit_manifest(df_: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    def add(check, value, status, detail=""):
        rows.append({"check": check, "value": value, "status": status, "detail": detail})

    required = [CLIP_ID_COL, GROUP_ID_COL, AUDIO_COL, SPLIT_COL] + LABEL_COLS
    missing = [c for c in required if c not in df_.columns]
    add("required_columns_missing", len(missing), "PASS" if not missing else "FAIL", ", ".join(missing))

    split_counts = df_[SPLIT_COL].value_counts(dropna=False).to_dict()
    add("split_counts", json.dumps(split_counts), "PASS", "")

    audio_exists = df_[AUDIO_COL].apply(lambda s: Path(str(s)).exists())
    add("audio_exists", f"{int(audio_exists.sum())}/{len(audio_exists)}", "PASS" if audio_exists.all() else "FAIL", "")

    label_nan = int(df_[LABEL_COLS].isna().sum().sum())
    add("label_nan_count", label_nan, "PASS" if label_nan == 0 else "FAIL", "")

    clip_unique = df_[CLIP_ID_COL].is_unique
    add("clip_id_unique", bool(clip_unique), "PASS" if clip_unique else "FAIL", "")

    subsets = {name: set(df_.loc[df_[SPLIT_COL] == name, GROUP_ID_COL].astype(str)) for name in [TRAIN_SPLIT_VALUE, VAL_SPLIT_VALUE, TEST_SPLIT_VALUE]}
    overlaps = {
        "train_val": len(subsets[TRAIN_SPLIT_VALUE] & subsets[VAL_SPLIT_VALUE]),
        "train_test": len(subsets[TRAIN_SPLIT_VALUE] & subsets[TEST_SPLIT_VALUE]),
        "val_test": len(subsets[VAL_SPLIT_VALUE] & subsets[TEST_SPLIT_VALUE]),
    }
    add("group_id_overlap", json.dumps(overlaps), "PASS" if sum(overlaps.values()) == 0 else "FAIL", "")

    clip_subsets = {name: set(df_.loc[df_[SPLIT_COL] == name, CLIP_ID_COL].astype(str)) for name in [TRAIN_SPLIT_VALUE, VAL_SPLIT_VALUE, TEST_SPLIT_VALUE]}
    clip_overlaps = {
        "train_val": len(clip_subsets[TRAIN_SPLIT_VALUE] & clip_subsets[VAL_SPLIT_VALUE]),
        "train_test": len(clip_subsets[TRAIN_SPLIT_VALUE] & clip_subsets[TEST_SPLIT_VALUE]),
        "val_test": len(clip_subsets[VAL_SPLIT_VALUE] & clip_subsets[TEST_SPLIT_VALUE]),
    }
    add("clip_id_overlap", json.dumps(clip_overlaps), "PASS" if sum(clip_overlaps.values()) == 0 else "FAIL", "")

    audit_df = pd.DataFrame(rows)
    save_csv(audit_df, out_dir / "strict_split_audit.csv")
    # WavLM posthoc-compatible alias.
    save_csv(audit_df, out_dir / "strict_split_leakage_audit.csv")
    display(audit_df)
    if (audit_df["status"] == "FAIL").any():
        raise RuntimeError("Audit failed. Fix data/manifest before continuing.")
    return audit_df


if MANIFEST_VAST.exists():
    log_line(f"Existing manifest_strict_vast.csv found: {MANIFEST_VAST}", LIVE_LOG)
    df = pd.read_csv(MANIFEST_VAST)
    df = normalize_split_values(df)
    try:
        audit_df = audit_manifest(df, OUT_TABLES)
    except RuntimeError as e:
        log_line(f"Existing manifest_strict_vast.csv audit failed: {repr(e)}", LIVE_LOG)
        if MANIFEST.exists():
            log_line("Rebuilding manifest_strict_vast.csv from manifest_strict.csv and available audio files.", LIVE_LOG)
            df = build_manifest_vast(force_rebuild=True)
            audit_df = audit_manifest(df, OUT_TABLES)
        else:
            raise
else:
    log_line("manifest_strict_vast.csv not found; building from manifest_strict.csv and available audio files.", LIVE_LOG)
    df = build_manifest_vast(force_rebuild=True)
    audit_df = audit_manifest(df, OUT_TABLES)


[2026-05-28 20:03:08] 
Cell | Prepare manifest_strict_vast.csv and audit split


[2026-05-28 20:03:08] Existing manifest_strict_vast.csv found: /workspace/hubert_finetune/data/manifest_strict_vast.csv


,check,value,status,detail
0,required_columns_missing,0,PASS,
1,split_counts,"{""train"": 5936, ""test"": 2039, ""val"": 1999}",PASS,
2,audio_exists,9974/9974,PASS,
3,label_nan_count,0,PASS,
4,clip_id_unique,True,PASS,
5,group_id_overlap,"{""train_val"": 0, ""train_test"": 0, ""val_test"": 0}",PASS,
6,clip_id_overlap,"{""train_val"": 0, ""train_test"": 0, ""val_test"": 0}",PASS,


In [8]:
# =========================
# Load manifest_strict_vast.csv and split data
# =========================
section("Cell | Load manifest_strict_vast.csv and split data")

MANIFEST_VAST = DATA / "manifest_strict_vast.csv"
if not MANIFEST_VAST.exists():
    log_line("manifest_strict_vast.csv not found; trying to build from manifest_strict.csv and audio files.", LIVE_LOG)
    df = build_manifest_vast(force_rebuild=True)
else:
    df = pd.read_csv(MANIFEST_VAST)
    df = normalize_split_values(df)

required_cols = [CLIP_ID_COL, GROUP_ID_COL, AUDIO_COL, SPLIT_COL] + LABEL_COLS
for c in required_cols:
    assert c in df.columns, f"Missing col: {c}"

audit_df = audit_manifest(df, OUT_TABLES)

df_train = df[df[SPLIT_COL] == TRAIN_SPLIT_VALUE].copy().reset_index(drop=True)
df_val = df[df[SPLIT_COL] == VAL_SPLIT_VALUE].copy().reset_index(drop=True)
df_test = df[df[SPLIT_COL] == TEST_SPLIT_VALUE].copy().reset_index(drop=True)

assert len(df_train) > 0, "df_train is empty"
assert len(df_val) > 0, "df_val is empty"
assert len(df_test) > 0, "df_test is empty"

log_line(f"train/val/test: {len(df_train)} / {len(df_val)} / {len(df_test)}", LIVE_LOG)
display(df[[CLIP_ID_COL, GROUP_ID_COL, SPLIT_COL, AUDIO_COL] + LABEL_COLS].head())


[2026-05-28 20:03:08] 
Cell | Load manifest_strict_vast.csv and split data


,check,value,status,detail
0,required_columns_missing,0,PASS,
1,split_counts,"{""train"": 5936, ""test"": 2039, ""val"": 1999}",PASS,
2,audio_exists,9974/9974,PASS,
3,label_nan_count,0,PASS,
4,clip_id_unique,True,PASS,
5,group_id_overlap,"{""train_val"": 0, ""train_test"": 0, ""val_test"": 0}",PASS,
6,clip_id_overlap,"{""train_val"": 0, ""train_test"": 0, ""val_test"": 0}",PASS,


[2026-05-28 20:03:09] train/val/test: 5936 / 1999 / 2039


,clip_id,group_id,split_strict,audio_path,extraversion,neuroticism,agreeableness,conscientiousness,openness
0,-10-QQDO_ME.001,-10-QQDO_ME,test,/workspace/hubert_finetune/data/preprocessed_f...,0.532710,0.416667,0.549451,0.563107,0.655556
1,-10-QQDO_ME.002,-10-QQDO_ME,test,/workspace/hubert_finetune/data/preprocessed_f...,0.579439,0.531250,0.670330,0.582524,0.655556
2,-10-QQDO_ME.005,-10-QQDO_ME,test,/workspace/hubert_finetune/data/preprocessed_f...,0.532710,0.552083,0.516484,0.611650,0.633333
3,-IK--4uz5ZY.000,-IK--4uz5ZY,test,/workspace/hubert_finetune/data/preprocessed_f...,0.579439,0.562500,0.604396,0.679612,0.588889
4,-IK--4uz5ZY.002,-IK--4uz5ZY,test,/workspace/hubert_finetune/data/preprocessed_f...,0.542056,0.635417,0.527473,0.660194,0.566667


In [9]:
# =========================
# Dataset and DataLoader
# =========================
section("Cell | Dataset and DataLoader")

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)


def trim_pad_wav(wav: np.ndarray) -> np.ndarray:
    if wav.shape[0] > MAX_LEN:
        return wav[:MAX_LEN]
    if wav.shape[0] < MAX_LEN:
        return np.pad(wav, (0, MAX_LEN - wav.shape[0]), mode="constant")
    return wav


class StrictAudioDataset(Dataset):
    def __init__(self, df_: pd.DataFrame):
        self.df = df_.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        audio_path = Path(str(row[AUDIO_COL]))
        try:
            wav, sr = sf.read(audio_path)
        except Exception as e:
            raise RuntimeError(f"Failed to read audio: {audio_path} | {repr(e)}")

        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        wav = wav.astype(np.float32)

        if sr != SR_TARGET:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
            wav = wav.astype(np.float32)

        wav = trim_pad_wav(wav).astype(np.float32)
        y = np.array([row[c] for c in LABEL_COLS], dtype=np.float32)
        clip_id = str(row[CLIP_ID_COL])
        return wav, y, clip_id


def collate_fn(batch):
    wavs, ys, clip_ids = zip(*batch)
    feats = feature_extractor(
        list(wavs),
        sampling_rate=SR_TARGET,
        return_tensors="pt",
        padding=True,
    )
    y = torch.tensor(np.stack(ys), dtype=torch.float32)
    return feats, y, list(clip_ids)


def make_loaders(df_tr: pd.DataFrame, df_va: pd.DataFrame, batch_size: int):
    train_ds = StrictAudioDataset(df_tr)
    val_ds = StrictAudioDataset(df_va)
    train_dl = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        collate_fn=collate_fn,
    )
    val_dl = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        collate_fn=collate_fn,
    )
    return train_dl, val_dl


def make_test_loader(df_te: pd.DataFrame, batch_size: int):
    test_ds = StrictAudioDataset(df_te)
    test_dl = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        collate_fn=collate_fn,
    )
    return test_dl

# Smoke check one batch.
_tmp_train_dl, _tmp_val_dl = make_loaders(df_train.head(min(len(df_train), BATCH_SIZE)), df_val.head(min(len(df_val), BATCH_SIZE)), batch_size=min(BATCH_SIZE, len(df_train)))
_inputs, _y, _clip_ids = next(iter(_tmp_train_dl))
log_line(f"Batch input_values: {tuple(_inputs['input_values'].shape)} | y: {tuple(_y.shape)} | first clip: {_clip_ids[0]}", LIVE_LOG)
del _tmp_train_dl, _tmp_val_dl, _inputs, _y, _clip_ids
gpu_flush()


[2026-05-28 20:03:09] 
Cell | Dataset and DataLoader


[2026-05-28 20:03:09] Batch input_values: (40, 240000) | y: (40, 5) | first clip: -N6QKrbnaDs.001


In [10]:
# =========================
# HuBERT + LoRA + regression head
# =========================
section("Cell | HuBERT + LoRA + regression head")


def list_lora_target_candidates(backbone_name: str) -> pd.DataFrame:
    base = AutoModel.from_pretrained(backbone_name)
    rows = []
    for name, module in base.named_modules():
        if any(name.endswith(t) for t in ["q_proj", "v_proj", "k_proj", "out_proj"]):
            rows.append({"module_name": name, "module_type": type(module).__name__})
    del base
    gpu_flush()
    return pd.DataFrame(rows)


target_df = list_lora_target_candidates(MODEL_NAME)
save_csv(target_df, OUT_TABLES / "hubert_lora_target_module_candidates.csv")
display(target_df.head(20))
assert target_df["module_name"].str.endswith("q_proj").any(), "No q_proj module found in HuBERT backbone."
assert target_df["module_name"].str.endswith("v_proj").any(), "No v_proj module found in HuBERT backbone."


class HuBERTWithHead(nn.Module):
    def __init__(
        self,
        backbone_name: str,
        r: int,
        lora_alpha: int = 32,
        lora_dropout: float = 0.05,
        head_dropout: float = 0.1,
    ):
        super().__init__()
        base = AutoModel.from_pretrained(backbone_name)
        for p in base.parameters():
            p.requires_grad = False

        lora_cfg = LoraConfig(
            r=r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["q_proj", "v_proj"],
            bias="none",
        )
        self.backbone = get_peft_model(base, lora_cfg)
        hidden = base.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, len(LABEL_COLS)),
            nn.Sigmoid(),
        )

    def mean_pool(self, x, attn_mask=None):
        if attn_mask is None:
            return x.mean(dim=1)
        m = attn_mask.unsqueeze(-1).type_as(x)
        return (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)

    def _get_feat_mask_safe(self, hidden_states, attention_mask):
        try:
            base_model = self.backbone.get_base_model() if hasattr(self.backbone, "get_base_model") else self.backbone
            if hasattr(base_model, "_get_feature_vector_attention_mask"):
                return base_model._get_feature_vector_attention_mask(hidden_states.shape[1], attention_mask)
        except Exception:
            pass
        return None

    def forward(self, input_values, attention_mask=None):
        out = self.backbone(input_values=input_values, attention_mask=attention_mask)
        h = out.last_hidden_state
        feat_mask = None
        if attention_mask is not None:
            feat_mask = self._get_feat_mask_safe(h, attention_mask)
        pooled = self.mean_pool(h, feat_mask)
        return self.head(pooled)


def trainable_params(model: nn.Module):
    return [p for p in model.parameters() if p.requires_grad]


def count_params(model: nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Model smoke test.
_tmp_model = HuBERTWithHead(MODEL_NAME, r=8).to(DEVICE)
_total, _trainable = count_params(_tmp_model)
log_line(f"Total params: {_total:,}", LIVE_LOG)
log_line(f"Trainable params: {_trainable:,} ({100*_trainable/max(_total,1):.4f}%)", LIVE_LOG)
_tmp_model.eval()
with torch.no_grad():
    _tmp_dl = make_test_loader(df_val.head(min(len(df_val), BATCH_SIZE)), batch_size=min(BATCH_SIZE, len(df_val)))
    _inputs, _y, _clip_ids = next(iter(_tmp_dl))
    _iv = _inputs["input_values"].to(DEVICE)
    _am = _inputs.get("attention_mask", None)
    if _am is not None:
        _am = _am.to(DEVICE)
    _out = _tmp_model(_iv, _am)
    log_line(f"Forward output shape: {tuple(_out.shape)}", LIVE_LOG)
    assert tuple(_out.shape) == (len(_clip_ids), len(LABEL_COLS))

del _tmp_model, _tmp_dl, _inputs, _y, _clip_ids, _iv, _am, _out
gpu_flush()


[2026-05-28 20:03:10] 
Cell | HuBERT + LoRA + regression head


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

,module_name,module_type
0,encoder.layers.0.attention.k_proj,Linear
1,encoder.layers.0.attention.v_proj,Linear
2,encoder.layers.0.attention.q_proj,Linear
3,encoder.layers.0.attention.out_proj,Linear
4,encoder.layers.1.attention.k_proj,Linear
5,encoder.layers.1.attention.v_proj,Linear
6,encoder.layers.1.attention.q_proj,Linear
7,encoder.layers.1.attention.out_proj,Linear
8,encoder.layers.2.attention.k_proj,Linear
9,encoder.layers.2.attention.v_proj,Linear


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[2026-05-28 20:03:12] Total params: 94,864,773


[2026-05-28 20:03:12] Trainable params: 493,061 (0.5198%)


[2026-05-28 20:03:13] Forward output shape: (40, 5)


In [11]:
# =========================
# Metrics
# =========================
section("Cell | Metrics")


def compute_metrics_from_arrays(y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
    mae_per = np.mean(np.abs(y_pred - y_true), axis=0)
    mae_mean = float(np.mean(mae_per))
    rmse_per = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    rmse_mean = float(np.mean(rmse_per))

    r2_per = []
    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]
        sst = float(np.sum((yt - np.mean(yt)) ** 2))
        if sst <= 1e-12:
            r2_per.append(np.nan)
        else:
            r2_per.append(float(r2_score(yt, yp)))
    r2_per = np.array(r2_per, dtype=float)
    r2_mean = float(np.nanmean(r2_per))

    acc_per = 1.0 - mae_per
    acc_mean = float(np.mean(acc_per))
    S = 1.0 - mae_mean
    return {
        "mae_per": mae_per,
        "mae_mean": mae_mean,
        "rmse_per": rmse_per,
        "rmse_mean": rmse_mean,
        "r2_per": r2_per,
        "r2_mean": r2_mean,
        "acc_per": acc_per,
        "acc_mean": acc_mean,
        "S": S,
    }


def metrics_to_flat_dict(metrics: Dict) -> Dict:
    out = {
        "mae_mean": float(metrics["mae_mean"]),
        "rmse_mean": float(metrics["rmse_mean"]),
        "r2_mean": float(metrics["r2_mean"]),
        "acc_mean": float(metrics["acc_mean"]),
        "S": float(metrics["S"]),
    }
    for i, col in enumerate(LABEL_COLS):
        out[f"mae_{col}"] = float(metrics["mae_per"][i])
        out[f"rmse_{col}"] = float(metrics["rmse_per"][i])
        out[f"r2_{col}"] = float(metrics["r2_per"][i]) if not np.isnan(metrics["r2_per"][i]) else np.nan
        out[f"acc_{col}"] = float(metrics["acc_per"][i])
    return out


def per_trait_metrics_df(y_true: np.ndarray, y_pred: np.ndarray, tag: str) -> pd.DataFrame:
    metrics = compute_metrics_from_arrays(y_true, y_pred)
    rows = []
    for i, col in enumerate(LABEL_COLS):
        rows.append({
            "tag": tag,
            "trait": col,
            "MAE": float(metrics["mae_per"][i]),
            "RMSE": float(metrics["rmse_per"][i]),
            "R2": float(metrics["r2_per"][i]),
            "Acc(1-MAE)": float(metrics["acc_per"][i]),
        })
    rows.append({
        "tag": tag,
        "trait": "mean",
        "MAE": float(metrics["mae_mean"]),
        "RMSE": float(metrics["rmse_mean"]),
        "R2": float(metrics["r2_mean"]),
        "Acc(1-MAE)": float(metrics["acc_mean"]),
    })
    return pd.DataFrame(rows)


[2026-05-28 20:03:14] 
Cell | Metrics


In [12]:
# =========================
# Plot helpers
# =========================
section("Cell | Plot helpers")


def save_and_maybe_show(fig, path: Path, show: bool = False):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


def plot_history(history_df: pd.DataFrame, out_dir: Path, show: bool = False):
    """Save WavLM-compatible per-run curves plus the combined train/validation loss curve."""
    if len(history_df) == 0:
        return

    # Combined train loss and validation loss in one figure.
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
    if "val_mae_mean" in history_df.columns:
        ax.plot(history_df["epoch"], history_df["val_mae_mean"], marker="o", label="validation_loss / val_MAE_mean")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MAE / L1 loss")
    ax.set_title("Train Loss and Validation Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_and_maybe_show(fig, out_dir / "loss_curve.png", show=show)

    # WavLM-compatible separate train loss curve.
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history_df["epoch"], history_df["train_loss"], marker="o")
    ax.set_title("Train Loss per Epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
    save_and_maybe_show(fig, out_dir / "curve_train_loss.png", show=show)

    # WavLM-compatible validation metric curves.
    for col, title, fname in [
        ("val_S", "Validation S per Epoch", "curve_val_S.png"),
        ("val_mae_mean", "Validation MAE Mean per Epoch", "curve_val_mae_mean.png"),
        ("val_rmse_mean", "Validation RMSE Mean per Epoch", "curve_val_rmse_mean.png"),
        ("val_r2_mean", "Validation R2 Mean per Epoch", "curve_val_r2_mean.png"),
    ]:
        if col not in history_df.columns:
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(history_df["epoch"], history_df[col], marker="o")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
        save_and_maybe_show(fig, out_dir / fname, show=show)

    # WavLM-compatible validation MAE per trait.
    trait_cols = [f"val_mae_{col}" for col in LABEL_COLS if f"val_mae_{col}" in history_df.columns]
    if trait_cols:
        fig, ax = plt.subplots(figsize=(9, 5))
        for col in LABEL_COLS:
            metric_col = f"val_mae_{col}"
            if metric_col in history_df.columns:
                ax.plot(history_df["epoch"], history_df[metric_col], marker="o", label=col)
        ax.set_title("Validation MAE per Trait")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MAE")
        ax.legend()
        ax.grid(True, alpha=0.3)
        save_and_maybe_show(fig, out_dir / "curve_mae_per_trait.png", show=show)


def plot_pred_vs_true(df_pred: pd.DataFrame, out_dir: Path, tag: str, show: bool = False):
    # Histogram true vs pred per trait.
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, col in enumerate(LABEL_COLS):
        ax = axes[i]
        ax.hist(df_pred[f"true_{col}"].values, bins=30, alpha=0.55, label="true")
        ax.hist(df_pred[f"pred_{col}"].values, bins=30, alpha=0.55, label="pred")
        ax.set_title(col)
        ax.set_xlabel("Score")
        ax.set_ylabel("Count")
        ax.legend()
    axes[-1].axis("off")
    save_and_maybe_show(fig, out_dir / f"hist_{tag}_pred_vs_true.png", show=show)

    # Scatter true vs pred per trait.
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, col in enumerate(LABEL_COLS):
        ax = axes[i]
        ax.scatter(df_pred[f"true_{col}"].values, df_pred[f"pred_{col}"].values, s=8, alpha=0.35)
        ax.plot([0, 1], [0, 1], linestyle="--")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(col)
        ax.set_xlabel("True")
        ax.set_ylabel("Pred")
    axes[-1].axis("off")
    save_and_maybe_show(fig, out_dir / f"scatter_{tag}_pred_vs_true.png", show=show)

    # WavLM-compatible residual scatter plot.
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, col in enumerate(LABEL_COLS):
        ax = axes[i]
        pred = df_pred[f"pred_{col}"].values
        residual = df_pred[f"pred_{col}"].values - df_pred[f"true_{col}"].values
        ax.scatter(pred, residual, s=8, alpha=0.35)
        ax.axhline(0, linestyle="--")
        ax.set_title(col)
        ax.set_xlabel("Pred")
        ax.set_ylabel("Residual")
    axes[-1].axis("off")
    save_and_maybe_show(fig, out_dir / f"scatter_{tag}_residuals.png", show=show)


def plot_bar_metrics(df_metric: pd.DataFrame, metric: str, out_path: Path, title: str, show: bool = False):
    d = df_metric[df_metric["trait"] != "mean"].copy()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(d["trait"], d[metric])
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=30)
    save_and_maybe_show(fig, out_path, show=show)


[2026-05-28 20:03:14] 
Cell | Plot helpers


In [13]:
# =========================
# Predict and evaluate
# =========================
section("Cell | Predict and evaluate")


def predict_and_evaluate(
    model: nn.Module,
    dl: DataLoader,
    run_id: str,
    seed: int,
    best_epoch: Optional[int] = None,
    best_val_S: Optional[float] = None,
    desc: str = "eval",
):
    model.eval()
    y_true_all = []
    y_pred_all = []
    clip_ids_all = []

    amp_ctx = torch.cuda.amp.autocast if AMP_ENABLED else nullcontext
    with torch.no_grad():
        pbar = tqdm(dl, desc=desc, leave=False, file=sys.stdout, dynamic_ncols=True)
        for inputs, y, clip_ids in pbar:
            iv = inputs["input_values"].to(DEVICE, non_blocking=True)
            am = inputs.get("attention_mask", None)
            if am is not None:
                am = am.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with amp_ctx():
                yhat = model(iv, am)
            y_true_all.append(y.detach().cpu().numpy())
            y_pred_all.append(yhat.detach().cpu().numpy())
            clip_ids_all.extend(clip_ids)
        pbar.close()

    y_true = np.concatenate(y_true_all, axis=0)
    y_pred = np.concatenate(y_pred_all, axis=0)
    metrics = compute_metrics_from_arrays(y_true, y_pred)

    rows = []
    for i, clip_id in enumerate(clip_ids_all):
        row = {
            "clip_id": clip_id,
            "run_id": run_id,
            "seed": seed,
        }
        if best_epoch is not None:
            row["best_epoch"] = int(best_epoch)
        if best_val_S is not None:
            row["best_val_S"] = float(best_val_S)
        for j, col in enumerate(LABEL_COLS):
            row[f"true_{col}"] = float(y_true[i, j])
            row[f"pred_{col}"] = float(y_pred[i, j])
            row[f"abs_err_{col}"] = float(abs(y_pred[i, j] - y_true[i, j]))
        row["mae_per_sample"] = float(np.mean([row[f"abs_err_{c}"] for c in LABEL_COLS]))
        rows.append(row)

    df_pred = pd.DataFrame(rows)
    return metrics, df_pred


[2026-05-28 20:03:14] 
Cell | Predict and evaluate


In [14]:
# =========================
# Optimizer, scheduler, checkpoint, train epoch
# =========================
section("Cell | Optimizer, scheduler, checkpoint, train epoch")


def build_optimizer(model: nn.Module, lr: float, weight_decay: float):
    return torch.optim.AdamW(trainable_params(model), lr=lr, weight_decay=weight_decay)


def build_scheduler(optimizer, train_loader_len: int, warmup_ratio: float):
    total_steps = max(1, train_loader_len * MAX_EPOCHS)
    warmup_steps = int(total_steps * warmup_ratio)
    return get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )


def save_checkpoint(ckpt_dir: Path, model: nn.Module, optimizer, scheduler, state: Dict):
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / "best_model.pt"
    payload = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        **state,
    }
    torch.save(payload, ckpt_path)
    save_json(state, ckpt_dir / "checkpoint_meta.json")
    return ckpt_path


def train_one_epoch(model, dl, optimizer, scheduler=None, desc="train"):
    model.train()
    total_loss = 0.0
    n = 0
    amp_ctx = torch.cuda.amp.autocast if AMP_ENABLED else nullcontext
    # Match the WavLM notebook behavior: initialize GradScaler inside each epoch.
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(dl, desc=desc, leave=False, file=sys.stdout, dynamic_ncols=True)
    for inputs, y, _clip_ids in pbar:
        iv = inputs["input_values"].to(DEVICE, non_blocking=True)
        am = inputs.get("attention_mask", None)
        if am is not None:
            am = am.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with amp_ctx():
            yhat = model(iv, am)
            loss = F.l1_loss(yhat, y, reduction="mean")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params(model), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        if scheduler is not None:
            scheduler.step()

        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        n += bs
        pbar.set_postfix(loss=f"{float(loss.item()):.5f}", avg=f"{float(total_loss/max(n,1)):.5f}")
    pbar.close()
    return float(total_loss / max(n, 1))


[2026-05-28 20:03:14] 
Cell | Optimizer, scheduler, checkpoint, train epoch


In [15]:
# =========================
# Single run training
# =========================
section("Cell | Single run training function")


def run_single_training(
    run_name: str,
    run_output_dir: Path,
    run_ckpt_dir: Path,
    df_tr: pd.DataFrame,
    df_va: pd.DataFrame,
    df_te: Optional[pd.DataFrame],
    seed: int,
    hp: Dict,
    is_optuna: bool = False,
    trial: Optional[optuna.Trial] = None,
    show_plots: bool = False,
):
    set_seed(seed)
    gpu_flush()
    run_output_dir.mkdir(parents=True, exist_ok=True)
    run_ckpt_dir.mkdir(parents=True, exist_ok=True)

    log_line(f"[{run_name}] START | seed={seed} | hp={hp}", LIVE_LOG)
    train_dl, val_dl = make_loaders(df_tr, df_va, batch_size=BATCH_SIZE)
    test_dl = make_test_loader(df_te, batch_size=BATCH_SIZE) if df_te is not None else None

    model = HuBERTWithHead(
        backbone_name=MODEL_NAME,
        r=int(hp["r"]),
        lora_alpha=int(hp["lora_alpha"]),
        lora_dropout=float(hp["lora_dropout"]),
        head_dropout=HEAD_DROPOUT,
    ).to(DEVICE)

    optimizer = build_optimizer(model, lr=float(hp["learning_rate"]), weight_decay=float(hp["weight_decay"]))
    scheduler = build_scheduler(optimizer, len(train_dl), warmup_ratio=float(hp["warmup_ratio"]))
    total_params, train_params = count_params(model)

    best_S = -1e18
    best_epoch = 0
    stop_epoch = 0
    patience_count = 0
    best_metrics_val = None
    best_ckpt_path = None
    best_df_pred_val = None
    last_metrics_val = None
    last_df_pred_val = None
    history_rows = []

    try:
        epoch_iter = tqdm(range(1, MAX_EPOCHS + 1), desc=f"[{run_name}] epochs", leave=False, file=sys.stdout, dynamic_ncols=True)
        for epoch in epoch_iter:
            train_loss = train_one_epoch(model, train_dl, optimizer, scheduler=scheduler, desc=f"{run_name} train e{epoch:03d}")
            val_metrics, df_pred_val = predict_and_evaluate(model, val_dl, run_id=run_name, seed=seed, desc=f"{run_name} val e{epoch:03d}")

            last_metrics_val = val_metrics
            last_df_pred_val = df_pred_val.copy()
            row = {"epoch": int(epoch), "train_loss": float(train_loss)}
            row.update({f"val_{k}": v for k, v in metrics_to_flat_dict(val_metrics).items()})
            history_rows.append(row)

            current_S = float(val_metrics["S"])
            if not np.isfinite(current_S):
                raise RuntimeError(f"{run_name} failed because val S is NaN/Inf at epoch {epoch}")

            improved = current_S > (best_S + MIN_DELTA)
            if improved:
                best_S = current_S
                best_epoch = int(epoch)
                patience_count = 0
                best_metrics_val = val_metrics
                best_df_pred_val = df_pred_val.copy()
                state = {
                    "run_name": run_name,
                    "seed": int(seed),
                    "best_epoch": int(best_epoch),
                    "best_val_S": float(best_S),
                    "total_params": int(total_params),
                    "trainable_params": int(train_params),
                    "hp": hp,
                    "is_optuna": bool(is_optuna),
                    "model_name": MODEL_NAME,
                    "experiment_name": EXPERIMENT_NAME,
                }
                best_ckpt_path = save_checkpoint(run_ckpt_dir, model, optimizer, scheduler, state)
            else:
                patience_count += 1

            log_line(
                f"[{run_name}] epoch={epoch:03d} | train_loss={train_loss:.6f} | "
                f"val_MAE={val_metrics['mae_mean']:.6f} | val_S={current_S:.6f} | "
                f"best_S={best_S:.6f} @epoch={best_epoch} | patience={patience_count}/{PATIENCE}",
                LIVE_LOG,
            )

            if trial is not None:
                trial.report(best_S, step=epoch)
                if trial.should_prune():
                    log_line(f"[{run_name}] PRUNED at epoch={epoch} | best_S={best_S:.6f}", LIVE_LOG)
                    raise optuna.TrialPruned()

            if patience_count >= PATIENCE:
                stop_epoch = int(epoch)
                log_line(f"[{run_name}] early stopping at epoch={stop_epoch}", LIVE_LOG)
                break

            del df_pred_val

        epoch_iter.close()
        if stop_epoch == 0:
            stop_epoch = int(history_rows[-1]["epoch"] if history_rows else MAX_EPOCHS)

        history_df = pd.DataFrame(history_rows)
        save_csv(history_df, run_output_dir / "history.csv")
        plot_history(history_df, run_output_dir, show=show_plots)

        if last_df_pred_val is not None:
            save_csv(last_df_pred_val, run_output_dir / "predictions_val.csv")
            plot_pred_vs_true(last_df_pred_val, run_output_dir, tag="val", show=False)
        if best_df_pred_val is not None:
            save_csv(best_df_pred_val, run_output_dir / "predictions_val_best.csv")
        if last_metrics_val is not None:
            save_json(metrics_to_flat_dict(last_metrics_val), run_output_dir / "last_val_metrics.json")
        if best_metrics_val is not None:
            save_json(metrics_to_flat_dict(best_metrics_val), run_output_dir / "best_val_metrics.json")

        assert best_ckpt_path is not None and Path(best_ckpt_path).exists(), f"Best checkpoint was not saved for {run_name}"
        ck = safe_torch_load(best_ckpt_path, map_location=DEVICE)
        model.load_state_dict(ck["model_state"], strict=True)

        run_summary = {
            "run_name": run_name,
            "seed": int(seed),
            "best_epoch": int(best_epoch),
            "stop_epoch": int(stop_epoch),
            "best_val_S": float(best_S),
            "total_params": int(total_params),
            "trainable_params": int(train_params),
            "checkpoint_path": str(best_ckpt_path),
            "hp": hp,
            "model_name": MODEL_NAME,
            "experiment_name": EXPERIMENT_NAME,
        }
        if best_metrics_val is not None:
            best_val_flat = metrics_to_flat_dict(best_metrics_val)
            run_summary.update({f"best_val_{k}": safe_to_float(v) for k, v in best_val_flat.items()})

        test_metrics = None
        df_pred_test = None
        if test_dl is not None:
            test_metrics, df_pred_test = predict_and_evaluate(
                model,
                test_dl,
                run_id=run_name,
                seed=seed,
                best_epoch=best_epoch,
                best_val_S=best_S,
                desc=f"{run_name} test",
            )
            save_csv(df_pred_test, run_output_dir / "predictions_test_strict.csv")
            plot_pred_vs_true(df_pred_test, run_output_dir, tag="test_strict", show=show_plots)
            test_flat = metrics_to_flat_dict(test_metrics)
            save_json(test_flat, run_output_dir / "test_metrics.json")
            run_summary.update({f"test_{k}": safe_to_float(v) for k, v in test_flat.items()})

            trait_df = per_trait_metrics_df(
                df_pred_test[[f"true_{c}" for c in LABEL_COLS]].values,
                df_pred_test[[f"pred_{c}" for c in LABEL_COLS]].values,
                tag=run_name,
            )
            save_csv(trait_df, run_output_dir / "test_per_trait_metrics.csv")

        save_json(run_summary, run_output_dir / "run_summary.json")
        log_line(f"[{run_name}] DONE | best_epoch={best_epoch} | best_val_S={best_S:.6f} | test_MAE={run_summary.get('test_mae_mean', np.nan):.6f} | test_R2={run_summary.get('test_r2_mean', np.nan):.6f}", LIVE_LOG)

        return {
            "run_summary": run_summary,
            "history_df": history_df,
            "best_ckpt_path": str(best_ckpt_path),
            "test_metrics": test_metrics,
            "df_pred_test": df_pred_test,
            "df_pred_val_best": best_df_pred_val,
            "df_pred_val_last": last_df_pred_val,
        }
    except optuna.TrialPruned:
        history_df = pd.DataFrame(history_rows)
        if len(history_df):
            save_csv(history_df, run_output_dir / "history.csv")
            plot_history(history_df, run_output_dir, show=False)
        raise
    except Exception as e:
        history_df = pd.DataFrame(history_rows)
        if len(history_df):
            save_csv(history_df, run_output_dir / "history.csv")
            plot_history(history_df, run_output_dir, show=False)
        save_json({"run_name": run_name, "seed": int(seed), "status": "failed", "error": repr(e), "hp": hp}, run_output_dir / "run_summary_failed.json")
        log_line(f"[{run_name}] FAILED | {repr(e)}", LIVE_LOG)
        raise
    finally:
        for obj_name in ["model", "optimizer", "scheduler", "scaler", "train_dl", "val_dl", "test_dl", "ck"]:
            if obj_name in locals():
                try:
                    del locals()[obj_name]
                except Exception:
                    pass
        gpu_flush()


[2026-05-28 20:03:14] 
Cell | Single run training function


In [16]:
# =========================
# Optuna search space and objective
# =========================
section("Cell | Optuna search space and objective")


def sample_hp(trial: optuna.Trial) -> Dict:
    # Same search space as WavLM for fair comparison.
    return {
        "learning_rate": trial.suggest_float("learning_rate", 5e-5, 3e-4, log=True),
        "r": trial.suggest_categorical("r", [4, 8, 16]),
        "lora_alpha": trial.suggest_categorical("lora_alpha", [16, 32, 64]),
        "lora_dropout": trial.suggest_categorical("lora_dropout", [0.0, 0.05, 0.1, 0.15]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-4, 2e-2, log=True),
        "warmup_ratio": trial.suggest_categorical("warmup_ratio", [0.0, 0.03, 0.05, 0.08, 0.1]),
    }


def objective(trial: optuna.Trial):
    trial_num = trial.number + 1
    trial_name = f"trial_{trial_num:03d}"
    trial_out_dir = OUT_OPTUNA_TRIALS / trial_name
    trial_ckpt_dir = CKPT_OPTUNA_TRIALS / trial_name
    hp = sample_hp(trial)
    log_line(f"[OPTUNA] {trial_name} start | hp={hp}", LIVE_LOG)

    try:
        result = run_single_training(
            run_name=trial_name,
            run_output_dir=trial_out_dir,
            run_ckpt_dir=trial_ckpt_dir,
            df_tr=df_train,
            df_va=df_val,
            df_te=None,
            seed=OPTUNA_SEED,
            hp=hp,
            is_optuna=True,
            trial=trial,
            show_plots=DISPLAY_TRIAL_PLOTS,
        )
        best_S = float(result["run_summary"]["best_val_S"])
        save_json({"trial_number": trial.number, "trial_name": trial_name, "value": best_S, "seed": OPTUNA_SEED, "hp": hp}, trial_out_dir / "trial_config.json")
        log_line(f"[OPTUNA] {trial_name} done | best_val_S={best_S:.6f}", LIVE_LOG)
        return best_S
    except optuna.TrialPruned:
        log_line(f"[OPTUNA] {trial_name} pruned", LIVE_LOG)
        raise
    except Exception as e:
        append_jsonl({"trial_name": trial_name, "error": repr(e), "hp": hp}, TRIAL_FAIL_LOG)
        log_line(f"[OPTUNA-FAIL] {trial_name} | {repr(e)}", LIVE_LOG)
        raise


[2026-05-28 20:03:14] 
Cell | Optuna search space and objective


In [17]:
# =========================
# Create/resume Optuna study and run optimization
# =========================
section("Cell | Run Optuna")

study_config = {
    "study_name": STUDY_NAME,
    "model_name": MODEL_NAME,
    "n_trials": N_TRIALS,
    "selection_metric": SELECTION_METRIC,
    "patience": PATIENCE,
    "batch_size": BATCH_SIZE,
    "split_source": str(MANIFEST_VAST),
    "pruner": str(PRUNER),
    "optuna_seed": OPTUNA_SEED,
    "note": "Optuna default sampler is used to mirror the WavLM notebook.",
}
save_json(study_config, OUT_OPTUNA_STUDY / "study_config.json")
save_json(study_config, OUT_TABLES / "study_config.json")

# Keep Optuna sampler behavior aligned with the WavLM notebook:
# no explicit sampler is passed to create_study, so Optuna uses its default sampler.
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="maximize",
    storage=STUDY_STORAGE,
    load_if_exists=True,
    pruner=PRUNER,
)

total_trials_before = len(study.trials)
completed_before = len([t for t in study.trials if t.state == TrialState.COMPLETE])
remaining = max(0, N_TRIALS - total_trials_before)
log_line(f"Study loaded: {STUDY_NAME} | total_trials_before={total_trials_before} | complete_before={completed_before} | remaining={remaining}", LIVE_LOG)

if remaining > 0:
    study.optimize(objective, n_trials=remaining, gc_after_trial=True, show_progress_bar=False)
else:
    log_line("No remaining Optuna trials. Reusing existing study.", LIVE_LOG)

# Save Optuna trial state counts for audit transparency
trial_state_counts = pd.Series(
    [str(t.state) for t in study.trials]
).value_counts().reset_index()

trial_state_counts.columns = ["state", "count"]

save_csv(trial_state_counts, OUT_OPTUNA_STUDY / "trial_state_counts.csv")
save_csv(trial_state_counts, OUT_TABLES / "trial_state_counts.csv")

complete_count = len([t for t in study.trials if t.state == TrialState.COMPLETE])

if complete_count < N_TRIALS:
    log_line(
        f"[WARN] Completed Optuna trials < N_TRIALS. "
        f"complete={complete_count}, total={len(study.trials)}. "
        "This matches WavLM total-trial policy, but must be reported.",
        LIVE_LOG,
    )

completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
assert len(completed) > 0, "No completed Optuna trial. Cannot continue."
best_trial = study.best_trial

completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
assert len(completed) > 0, "No completed Optuna trial. Cannot continue."
best_trial = study.best_trial
log_line(f"Best trial: #{best_trial.number} | value={best_trial.value:.6f} | params={best_trial.params}", LIVE_LOG)
best_trial_summary = {
    "number": int(best_trial.number),
    "value": float(best_trial.value),
    "params": dict(best_trial.params),
}
save_json(best_trial_summary, OUT_OPTUNA_STUDY / "best_trial_summary.json")
save_json(best_trial_summary, OUT_TABLES / "best_trial_summary.json")


[2026-05-28 20:03:14] 
Cell | Run Optuna


[2026-05-28 20:03:15] Study loaded: hubert_lora_strict_optuna | total_trials_before=0 | complete_before=0 | remaining=50


[2026-05-28 20:03:16] [OPTUNA] trial_001 start | hp={'learning_rate': 0.00012816842992683473, 'r': 4, 'lora_alpha': 64, 'lora_dropout': 0.15, 'weight_decay': 0.015177021393917542, 'warmup_ratio': 0.0}


[2026-05-28 20:03:16] [trial_001] START | seed=42 | hp={'learning_rate': 0.00012816842992683473, 'r': 4, 'lora_alpha': 64, 'lora_dropout': 0.15, 'weight_decay': 0.015177021393917542, 'warmup_ratio': 0.0}


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

trial_003 train e011:   7%|██████▎                                                                                       | 10/149 [00:07<01:39,  1.40it/s, avg=0.09474, loss=0.08962]

trial_003 train e011:   7%|██████▎                                                                                       | 10/149 [00:07<01:39,  1.40it/s, avg=0.09422, loss=0.08903]

trial_003 train e011:   7%|██████▉                                                                                       | 11/149 [00:07<01:38,  1.40it/s, avg=0.09422, loss=0.08903]

trial_003 train e011:   7%|██████▉                                                                                       | 11/149 [00:08<01:38,  1.40it/s, avg=0.09342, loss=0.08464]

trial_003 train e011:   8%|███████▌                                                                                      | 12/149 [00:08<01:37,  1.40it/s, avg=0.09342, loss=0.08464]

In [ ]:
# =========================
# Save study results and plots
# =========================
section("Cell | Save Optuna study results")

trials_df = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs", "datetime_start", "datetime_complete", "duration"))
save_csv(trials_df, OUT_OPTUNA_STUDY / "optuna_trials_dataframe.csv")
save_csv(trials_df, OUT_TABLES / "optuna_trials_dataframe.csv")
# WavLM-compatible alias.
save_csv(trials_df, OUT_OPTUNA_STUDY / "study_trials.csv")
save_csv(trials_df, OUT_TABLES / "study_trials.csv")
display(trials_df.tail(10))

# Completed trials summary.
complete_rows = []
for t in study.trials:
    if t.state == TrialState.COMPLETE:
        row = {"number": t.number, "value": t.value}
        row.update(t.params)
        complete_rows.append(row)
complete_df = pd.DataFrame(complete_rows).sort_values("value", ascending=False).reset_index(drop=True)
save_csv(complete_df, OUT_OPTUNA_STUDY / "optuna_complete_trials.csv")
save_csv(complete_df.head(OPTUNA_TOPK), OUT_OPTUNA_STUDY / "optuna_top_trials.csv")
save_csv(complete_df.head(OPTUNA_TOPK), OUT_TABLES / "optuna_top_trials.csv")
# WavLM-compatible alias.
save_csv(complete_df.head(OPTUNA_TOPK), OUT_OPTUNA_STUDY / "top5_trials.csv")
save_csv(complete_df.head(OPTUNA_TOPK), OUT_TABLES / "top5_trials.csv")
display(complete_df.head(OPTUNA_TOPK))

# Optimization history plot.
if len(complete_df) > 0:
    hist_rows = []
    best_so_far = -np.inf
    for t in sorted([t for t in study.trials if t.state == TrialState.COMPLETE], key=lambda x: x.number):
        best_so_far = max(best_so_far, float(t.value))
        hist_rows.append({"trial": t.number, "value": float(t.value), "best_so_far": float(best_so_far)})
    hist_df = pd.DataFrame(hist_rows)
    save_csv(hist_df, OUT_OPTUNA_STUDY / "optuna_optimization_history.csv")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(hist_df["trial"], hist_df["value"], marker="o", label="trial value")
    ax.plot(hist_df["trial"], hist_df["best_so_far"], marker="o", label="best so far")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Validation S = 1 - MAE")
    ax.set_title("Optuna Optimization History")
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_and_maybe_show(fig, OUT_OPTUNA_STUDY / "optuna_optimization_history.png", show=DISPLAY_SUMMARY_PLOTS)
    copy_if_exists(OUT_OPTUNA_STUDY / "optuna_optimization_history.png", OUT_FIGURES / "optuna_optimization_history.png")
# WavLM-compatible alias.
copy_if_exists(OUT_OPTUNA_STUDY / "optuna_optimization_history.png", OUT_OPTUNA_STUDY / "optimization_history.png")
copy_if_exists(OUT_OPTUNA_STUDY / "optuna_optimization_history.png", OUT_FIGURES / "optimization_history.png")

# Parameter importance plot.
try:
    importances = optuna.importance.get_param_importances(study)
    imp_df = pd.DataFrame([{"parameter": k, "importance": v} for k, v in importances.items()])
    save_csv(imp_df, OUT_OPTUNA_STUDY / "optuna_param_importance.csv")
    save_csv(imp_df, OUT_TABLES / "optuna_param_importance.csv")
    if len(imp_df) > 0:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.bar(imp_df["parameter"], imp_df["importance"])
        ax.set_title("Optuna Parameter Importance")
        ax.set_ylabel("Importance")
        ax.tick_params(axis="x", rotation=30)
        save_and_maybe_show(fig, OUT_OPTUNA_STUDY / "optuna_param_importance.png", show=DISPLAY_SUMMARY_PLOTS)
        copy_if_exists(OUT_OPTUNA_STUDY / "optuna_param_importance.png", OUT_FIGURES / "optuna_param_importance.png")
        # WavLM-compatible alias.
        copy_if_exists(OUT_OPTUNA_STUDY / "optuna_param_importance.png", OUT_OPTUNA_STUDY / "parameter_importance.png")
        copy_if_exists(OUT_OPTUNA_STUDY / "optuna_param_importance.png", OUT_FIGURES / "parameter_importance.png")
except Exception as e:
    log_line(f"Parameter importance skipped: {repr(e)}", LIVE_LOG)

# Keep only top-k Optuna checkpoints to control disk usage, following the WavLM notebook behavior.
keep_trial_numbers = set(complete_df.head(OPTUNA_TOPK)["number"].astype(int).tolist())
keep_trial_names = {f"trial_{n+1:03d}" for n in keep_trial_numbers}
save_csv(pd.DataFrame({"kept_trial_dir": sorted(keep_trial_names)}), OUT_OPTUNA_STUDY / "kept_optuna_checkpoint_dirs.csv")

removed_dirs = []
for ckpt_dir in sorted(CKPT_OPTUNA_TRIALS.glob("trial_*")):
    if ckpt_dir.name not in keep_trial_names:
        shutil.rmtree(ckpt_dir, ignore_errors=True)
        removed_dirs.append(ckpt_dir.name)
save_csv(pd.DataFrame({"removed_trial_dir": removed_dirs}), OUT_OPTUNA_STUDY / "removed_optuna_checkpoint_dirs.csv")
log_line(f"Kept Optuna checkpoint dirs: {sorted(keep_trial_names)}", LIVE_LOG)
log_line(f"Removed Optuna checkpoint dirs outside top-{OPTUNA_TOPK}: {len(removed_dirs)}", LIVE_LOG)

BEST_HP = {
    "learning_rate": float(best_trial.params["learning_rate"]),
    "r": int(best_trial.params["r"]),
    "lora_alpha": int(best_trial.params["lora_alpha"]),
    "lora_dropout": float(best_trial.params["lora_dropout"]),
    "weight_decay": float(best_trial.params["weight_decay"]),
    "warmup_ratio": float(best_trial.params["warmup_ratio"]),
}
save_json(BEST_HP, OUT_OPTUNA_STUDY / "best_params.json")
save_json(BEST_HP, OUT_OPTUNA_STUDY / "optuna_best_params.json")
save_json(BEST_HP, OUT_OPTUNA_STUDY / "best_hp.json")
copy_if_exists(OUT_OPTUNA_STUDY / "best_hp.json", OUT_TABLES / "best_hp.json")
save_csv(pd.DataFrame([BEST_HP]), OUT_OPTUNA_STUDY / "best_params.csv")
save_csv(pd.DataFrame([BEST_HP]), OUT_TABLES / "best_params.csv")
log_line(f"BEST_HP: {BEST_HP}", LIVE_LOG)


In [ ]:
# =========================
# Final runs
# =========================
section("Cell | Final runs")

def validate_existing_final_run(run_name: str, run_out_dir: Path) -> Optional[Dict]:
    summary_path = run_out_dir / "run_summary.json"
    pred_test_path = run_out_dir / "predictions_test_strict.csv"
    history_path = run_out_dir / "history.csv"
    ckpt_path = CKPT_FINAL_RUNS / run_name / "best_model.pt"

    if not (
        summary_path.exists()
        and pred_test_path.exists()
        and history_path.exists()
        and ckpt_path.exists()
    ):
        return None

    try:
        with open(summary_path, "r", encoding="utf-8") as f:
            rs = json.load(f)

        # =========================
        # Basic artifact validation
        # =========================
        df_pred_check = pd.read_csv(pred_test_path)

        if len(df_pred_check) != len(df_test):
            log_line(
                f"[FINAL-RESUME] {run_name} existing prediction row mismatch; rerun needed.",
                LIVE_LOG,
            )
            return None

        if not df_pred_check["clip_id"].is_unique:
            log_line(
                f"[FINAL-RESUME] {run_name} existing prediction clip_id not unique; rerun needed.",
                LIVE_LOG,
            )
            return None
        
        # Validate exact test clip_id set to prevent stale resume
        expected_clip_ids = set(df_test[CLIP_ID_COL].astype(str).tolist())
        actual_clip_ids = set(df_pred_check["clip_id"].astype(str).tolist())

        if actual_clip_ids != expected_clip_ids:
            missing = sorted(expected_clip_ids - actual_clip_ids)[:10]
            extra = sorted(actual_clip_ids - expected_clip_ids)[:10]
            log_line(
                f"[FINAL-RESUME] {run_name} test clip_id set mismatch; "
                f"missing_first10={missing}; extra_first10={extra}; rerun needed.",
                LIVE_LOG,
            )
            return None
        
        required_summary_keys = ["best_epoch", "stop_epoch", "best_val_S"]
        if any(k not in rs for k in required_summary_keys):
            log_line(
                f"[FINAL-RESUME] {run_name} existing summary incomplete; rerun needed.",
                LIVE_LOG,
            )
            return None

        # =========================
        # Prevent stale resume
        # =========================

        # 1. Validate model name if available in run_summary.json
        if "model_name" in rs and rs.get("model_name") != MODEL_NAME:
            log_line(
                f"[FINAL-RESUME] {run_name} model mismatch "
                f"({rs.get('model_name')} != {MODEL_NAME}); rerun needed.",
                LIVE_LOG,
            )
            return None

        # 2. Validate seed from run_name against summary seed
        # Expected run_name example: run_01_seed_042
        if "seed" in rs:
            try:
                expected_seed = int(str(run_name).split("_seed_")[-1])
                actual_seed = int(rs.get("seed"))
                if actual_seed != expected_seed:
                    log_line(
                        f"[FINAL-RESUME] {run_name} seed mismatch "
                        f"({actual_seed} != {expected_seed}); rerun needed.",
                        LIVE_LOG,
                    )
                    return None
            except Exception:
                log_line(
                    f"[FINAL-RESUME] {run_name} seed validation failed; rerun needed.",
                    LIVE_LOG,
                )
                return None

        # 3. Validate best hyperparameters if available in run_summary.json
        if "hp" in rs:
            for k, v in BEST_HP.items():
                if k not in rs["hp"]:
                    log_line(
                        f"[FINAL-RESUME] {run_name} missing HP key '{k}'; rerun needed.",
                        LIVE_LOG,
                    )
                    return None

                old_v = rs["hp"].get(k)

                # Compare floats safely, otherwise compare as string
                try:
                    if isinstance(v, float) or isinstance(old_v, float):
                        if not np.isclose(float(old_v), float(v), rtol=1e-9, atol=1e-12):
                            log_line(
                                f"[FINAL-RESUME] {run_name} HP mismatch at {k} "
                                f"({old_v} != {v}); rerun needed.",
                                LIVE_LOG,
                            )
                            return None
                    else:
                        if str(old_v) != str(v):
                            log_line(
                                f"[FINAL-RESUME] {run_name} HP mismatch at {k} "
                                f"({old_v} != {v}); rerun needed.",
                                LIVE_LOG,
                            )
                            return None
                except Exception:
                    if str(old_v) != str(v):
                        log_line(
                            f"[FINAL-RESUME] {run_name} HP mismatch at {k} "
                            f"({old_v} != {v}); rerun needed.",
                            LIVE_LOG,
                        )
                        return None

        # 4. Validate required prediction columns
        required_pred_cols = ["clip_id"]
        for c in LABEL_COLS:
            required_pred_cols.extend([
                f"true_{c}",
                f"pred_{c}",
                f"abs_err_{c}",
            ])

        missing_pred_cols = [c for c in required_pred_cols if c not in df_pred_check.columns]
        if missing_pred_cols:
            log_line(
                f"[FINAL-RESUME] {run_name} missing prediction columns "
                f"{missing_pred_cols}; rerun needed.",
                LIVE_LOG,
            )
            return None
        pred_cols = [f"pred_{c}" for c in LABEL_COLS]
        true_cols = [f"true_{c}" for c in LABEL_COLS]

        if df_pred_check[pred_cols + true_cols].isna().any().any():
            log_line(f"[FINAL-RESUME] {run_name} has NaN in prediction/target columns; rerun needed.", LIVE_LOG)
            return None

        if not ((df_pred_check[pred_cols] >= 0).all().all() and (df_pred_check[pred_cols] <= 1).all().all()):
            log_line(f"[FINAL-RESUME] {run_name} prediction values outside [0,1]; rerun needed.", LIVE_LOG)
            return None
        
        log_line(
            f"[FINAL-RESUME] {run_name} existing artifacts valid; resume accepted.",
            LIVE_LOG,
        )
        return rs

    except Exception as e:
        log_line(
            f"[FINAL-RESUME] {run_name} validation error: {repr(e)}; rerun needed.",
            LIVE_LOG,
        )
        return None

final_run_rows = []
for idx, seed in enumerate(FINAL_SEEDS, start=1):
    run_name = f"run_{idx:02d}_seed_{seed:03d}"
    run_out_dir = OUT_FINAL_RUNS / run_name
    run_ckpt_dir = CKPT_FINAL_RUNS / run_name

    rs = validate_existing_final_run(run_name, run_out_dir)
    if rs is not None:
        log_line(f"[FINAL-RESUME] using existing completed run: {run_name}", LIVE_LOG)
    else:
        log_line(f"[FINAL] start {run_name}", LIVE_LOG)
        result = run_single_training(
            run_name=run_name,
            run_output_dir=run_out_dir,
            run_ckpt_dir=run_ckpt_dir,
            df_tr=df_train,
            df_va=df_val,
            df_te=df_test,
            seed=seed,
            hp=BEST_HP,
            is_optuna=False,
            trial=None,
            show_plots=DISPLAY_FINAL_RUN_PLOTS,
        )
        rs = result["run_summary"]
        del result
        gpu_flush()

    pred_test_path = run_out_dir / "predictions_test_strict.csv"
    assert pred_test_path.exists(), f"Missing test prediction CSV: {pred_test_path}"
    df_pred_check = pd.read_csv(pred_test_path)
    assert len(df_pred_check) == len(df_test), f"Prediction rows mismatch in {run_name}: {len(df_pred_check)} vs {len(df_test)}"
    assert df_pred_check["clip_id"].is_unique, f"clip_id is not unique in {pred_test_path}"

    final_run_rows.append({
        "run_id": run_name,
        "seed": int(seed),
        "best_epoch": int(rs["best_epoch"]),
        "stop_epoch": int(rs["stop_epoch"]),
        "best_val_S": float(rs["best_val_S"]),
        "test_MAE_mean": rs.get("test_mae_mean", np.nan),
        "test_RMSE_mean": rs.get("test_rmse_mean", np.nan),
        "test_R2_mean": rs.get("test_r2_mean", np.nan),
        "test_Acc_mean": rs.get("test_acc_mean", np.nan),
        "checkpoint_path": rs.get("checkpoint_path"),
    })

    log_line(f"[FINAL] recorded {run_name} | test_MAE={rs.get('test_mae_mean', np.nan):.6f} | test_R2={rs.get('test_r2_mean', np.nan):.6f}", LIVE_LOG)
    del df_pred_check
    gpu_flush()

final_runs_summary = pd.DataFrame(final_run_rows)
save_csv(final_runs_summary, OUT_FINAL_AGG / "final_runs_summary.csv")
save_csv(final_runs_summary, OUT_TABLES / "final_runs_summary.csv")
display(final_runs_summary)


In [ ]:
# =========================
# Aggregate statistics, selection, and ensemble
# =========================
section("Cell | Aggregate statistics, selection, and ensemble")

summary = pd.read_csv(OUT_FINAL_AGG / "final_runs_summary.csv")
metrics_cols = ["best_val_S", "test_MAE_mean", "test_RMSE_mean", "test_R2_mean", "test_Acc_mean"]
stat_rows = []
for col in metrics_cols:
    vals = summary[col].dropna().astype(float).values
    if len(vals) == 0:
        continue
    mean = float(np.mean(vals))
    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    sem = std / math.sqrt(len(vals)) if len(vals) > 1 else 0.0
    ci95 = 1.96 * sem if len(vals) > 1 else 0.0
    stat_rows.append({"metric": col, "n": len(vals), "mean": mean, "std": std, "min": float(np.min(vals)), "median": float(np.median(vals)), "max": float(np.max(vals)), "ci95_half_width": ci95})
final_stats_long = pd.DataFrame(stat_rows)
save_csv(final_stats_long, OUT_FINAL_AGG / "final_stats_summary_long.csv")
save_csv(final_stats_long, OUT_TABLES / "final_stats_summary_long.csv")
save_json({r["metric"]: r for r in stat_rows}, OUT_FINAL_AGG / "final_stats_summary_long.json")
display(final_stats_long)

# =========================
# Extra WavLM-compatible statistical artifacts
# =========================

def summarize_series(x: pd.Series, prefix: str) -> Dict:
    arr = x.dropna().values.astype(float)
    if len(arr) == 0:
        return {}

    mean = float(arr.mean())
    std = float(arr.std(ddof=1)) if len(arr) > 1 else 0.0
    ci95 = float(1.96 * std / math.sqrt(len(arr))) if len(arr) > 1 else 0.0

    return {
        f"{prefix}_n": int(len(arr)),
        f"{prefix}_mean": mean,
        f"{prefix}_std": std,
        f"{prefix}_min": float(np.min(arr)),
        f"{prefix}_max": float(np.max(arr)),
        f"{prefix}_p10": float(np.percentile(arr, 10)),
        f"{prefix}_p50": float(np.percentile(arr, 50)),
        f"{prefix}_p90": float(np.percentile(arr, 90)),
        f"{prefix}_ci95_low": mean - ci95,
        f"{prefix}_ci95_high": mean + ci95,
        f"{prefix}_ci95_half_width": ci95,
    }


stats = {}

for col in [
    "best_val_S",
    "test_MAE_mean",
    "test_RMSE_mean",
    "test_R2_mean",
    "test_Acc_mean",
    "best_epoch",
    "stop_epoch",
]:
    if col in summary.columns:
        stats.update(summarize_series(summary[col], col))

stats_wide_df = pd.DataFrame([stats])
save_csv(stats_wide_df, OUT_FINAL_AGG / "final_stats_summary_wide.csv")
save_csv(stats_wide_df, OUT_TABLES / "final_stats_summary_wide.csv")
save_json(make_json_safe(stats), OUT_FINAL_AGG / "final_stats_summary.json")

# WavLM-compatible alias
save_csv(stats_wide_df, OUT_FINAL_AGG / "final_stats_summary.csv")
save_csv(stats_wide_df, OUT_TABLES / "final_stats_summary.csv")

# 95% CI report per metric
ci_rows = []
for col in [
    "best_val_S",
    "test_MAE_mean",
    "test_RMSE_mean",
    "test_R2_mean",
    "test_Acc_mean",
]:
    if col in summary.columns and f"{col}_mean" in stats:
        ci_rows.append({
            "metric": col,
            "n": stats.get(f"{col}_n"),
            "mean": stats.get(f"{col}_mean"),
            "std": stats.get(f"{col}_std"),
            "ci95_low": stats.get(f"{col}_ci95_low"),
            "ci95_high": stats.get(f"{col}_ci95_high"),
            "ci95_half_width": stats.get(f"{col}_ci95_half_width"),
        })

ci_df = pd.DataFrame(ci_rows)
save_csv(ci_df, OUT_FINAL_AGG / "ci_95_report.csv")
save_csv(ci_df, OUT_TABLES / "ci_95_report.csv")


# Epoch behavior report
epoch_cols = [c for c in ["run_id", "seed", "best_epoch", "stop_epoch"] if c in summary.columns]
epoch_behavior_df = summary[epoch_cols].copy()
save_csv(epoch_behavior_df, OUT_FINAL_AGG / "epoch_behavior.csv")
save_csv(epoch_behavior_df, OUT_TABLES / "epoch_behavior.csv")


# Validation stability report
stability_cols = [
    c for c in [
        "run_id",
        "seed",
        "best_val_S",
        "test_MAE_mean",
        "test_RMSE_mean",
        "test_R2_mean",
        "test_Acc_mean",
    ]
    if c in summary.columns
]
validation_stability_df = summary[stability_cols].copy()
save_csv(validation_stability_df, OUT_FINAL_AGG / "validation_stability.csv")
save_csv(validation_stability_df, OUT_TABLES / "validation_stability.csv")


# Best epoch distribution
if "best_epoch" in summary.columns:
    best_epoch_dist = (
        summary["best_epoch"]
        .value_counts()
        .sort_index()
        .reset_index()
    )
    best_epoch_dist.columns = ["best_epoch", "count"]
    save_csv(best_epoch_dist, OUT_FINAL_AGG / "best_epoch_distribution.csv")
    save_csv(best_epoch_dist, OUT_TABLES / "best_epoch_distribution.csv")


# Stop epoch distribution
if "stop_epoch" in summary.columns:
    stop_epoch_dist = (
        summary["stop_epoch"]
        .value_counts()
        .sort_index()
        .reset_index()
    )
    stop_epoch_dist.columns = ["stop_epoch", "count"]
    save_csv(stop_epoch_dist, OUT_FINAL_AGG / "stop_epoch_distribution.csv")
    save_csv(stop_epoch_dist, OUT_TABLES / "stop_epoch_distribution.csv")


log_line("[FINAL] Extra WavLM-compatible statistical artifacts saved.", LIVE_LOG)



# Simple metric curves across final runs.
for metric in ["test_MAE_mean", "test_R2_mean", "test_Acc_mean"]:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(1, len(summary)+1), summary[metric], marker="o")
    ax.set_xlabel("Run index")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} Across Final Runs")
    ax.grid(True, alpha=0.3)
    save_and_maybe_show(fig, OUT_FINAL_AGG / f"curve_{metric}_across_runs.png", show=DISPLAY_SUMMARY_PLOTS)
    copy_if_exists(OUT_FINAL_AGG / f"curve_{metric}_across_runs.png", OUT_FIGURES / f"curve_{metric}_across_runs.png")

# Select best-val run and median run.
best_val_row = summary.sort_values("best_val_S", ascending=False).iloc[0].to_dict()
# Match WavLM behavior: select representative median run using validation score, not test metric.
sorted_by_val = summary.sort_values("best_val_S", ascending=True).reset_index(drop=True)
median_row = sorted_by_val.iloc[len(sorted_by_val) // 2].to_dict()
selected_info = {"best_val": best_val_row, "median": median_row}
save_json(selected_info, OUT_FINAL_AGG / "selected_runs_summary.json")
save_csv(pd.DataFrame([best_val_row, median_row], index=["best_val", "median"]).reset_index(names="selection"), OUT_TABLES / "best_val_median_runs.csv")
display(pd.DataFrame([best_val_row, median_row], index=["best_val", "median"]))


def copy_selected_run(selection_name: str, row: Dict):
    src_run = OUT_FINAL_RUNS / str(row["run_id"])
    dst_run = OUT_FINAL_SELECTED / selection_name
    dst_run.mkdir(parents=True, exist_ok=True)
    for fname in [
        "history.csv", "loss_curve.png", "curve_train_loss.png", "curve_val_S.png",
        "curve_val_mae_mean.png", "curve_val_rmse_mean.png", "curve_val_r2_mean.png",
        "curve_mae_per_trait.png", "predictions_val_best.csv", "predictions_test_strict.csv",
        "test_metrics.json", "best_val_metrics.json", "test_per_trait_metrics.csv",
        "hist_test_strict_pred_vs_true.png", "scatter_test_strict_pred_vs_true.png",
        "scatter_test_strict_residuals.png"
    ]:
        copy_if_exists(src_run / fname, dst_run / fname)
    ckpt_src = CKPT_FINAL_RUNS / str(row["run_id"]) / "best_model.pt"
    ckpt_dst = CKPT_FINAL_SELECTED / selection_name / "best_model.pt"
    copy_if_exists(ckpt_src, ckpt_dst)

copy_selected_run("best_val", best_val_row)
copy_selected_run("median", median_row)

# Ensemble: average predictions from all final runs.
pred_paths = []
for run_id in summary["run_id"].tolist():
    p = OUT_FINAL_RUNS / str(run_id) / "predictions_test_strict.csv"
    assert p.exists(), f"Missing prediction: {p}"
    pred_paths.append(p)

base = pd.read_csv(pred_paths[0]).sort_values("clip_id").reset_index(drop=True)
ensemble_sources = [str(p) for p in pred_paths]
for col in LABEL_COLS:
    preds = []
    for p in pred_paths:
        d = pd.read_csv(p).sort_values("clip_id").reset_index(drop=True)
        assert (d["clip_id"].values == base["clip_id"].values).all(), f"clip_id order mismatch: {p}"
        preds.append(d[f"pred_{col}"].values)
    base[f"pred_{col}"] = np.mean(np.stack(preds, axis=0), axis=0)
    base[f"abs_err_{col}"] = np.abs(base[f"pred_{col}"] - base[f"true_{col}"])
base["run_id"] = "ensemble_mean_10runs"
base["seed"] = -1
base["mae_per_sample"] = base[[f"abs_err_{c}" for c in LABEL_COLS]].mean(axis=1)

y_true = base[[f"true_{c}" for c in LABEL_COLS]].values
y_pred = base[[f"pred_{c}" for c in LABEL_COLS]].values
ens_metrics = compute_metrics_from_arrays(y_true, y_pred)
ens_flat = metrics_to_flat_dict(ens_metrics)
ENS_OUT = OUT_FINAL_SELECTED / "ensemble"
ENS_OUT.mkdir(parents=True, exist_ok=True)
save_csv(base, ENS_OUT / "ensemble_predictions_test_strict.csv")
save_json(ens_flat, ENS_OUT / "ensemble_metrics.json")
save_json({"sources": ensemble_sources}, ENS_OUT / "ensemble_sources.json")
plot_pred_vs_true(base, ENS_OUT, tag="ensemble", show=DISPLAY_SUMMARY_PLOTS)
ens_trait = per_trait_metrics_df(y_true, y_pred, tag="ensemble")
save_csv(ens_trait, ENS_OUT / "ensemble_per_trait_metrics.csv")
save_csv(ens_trait, OUT_TABLES / "per_trait_metrics_ensemble.csv")
display(pd.DataFrame([ens_flat]))

# =========================================================
# Best-val, median, ensemble summary table
# =========================================================
bme_rows = []

bme_rows.append({
    "selection": "best_val",
    "run_id": best_val_row.get("run_id"),
    "seed": best_val_row.get("seed"),
    "best_val_S": best_val_row.get("best_val_S"),
    "test_MAE_mean": best_val_row.get("test_MAE_mean"),
    "test_RMSE_mean": best_val_row.get("test_RMSE_mean"),
    "test_R2_mean": best_val_row.get("test_R2_mean"),
    "test_Acc_mean": best_val_row.get("test_Acc_mean"),
})

bme_rows.append({
    "selection": "median",
    "run_id": median_row.get("run_id"),
    "seed": median_row.get("seed"),
    "best_val_S": median_row.get("best_val_S"),
    "test_MAE_mean": median_row.get("test_MAE_mean"),
    "test_RMSE_mean": median_row.get("test_RMSE_mean"),
    "test_R2_mean": median_row.get("test_R2_mean"),
    "test_Acc_mean": median_row.get("test_Acc_mean"),
})

bme_rows.append({
    "selection": "ensemble",
    "run_id": "ensemble_mean_10runs",
    "seed": -1,
    "best_val_S": np.nan,
    "test_MAE_mean": ens_flat.get("mae_mean"),
    "test_RMSE_mean": ens_flat.get("rmse_mean"),
    "test_R2_mean": ens_flat.get("r2_mean"),
    "test_Acc_mean": ens_flat.get("acc_mean"),
})

bme_df = pd.DataFrame(bme_rows)

save_csv(bme_df, OUT_FINAL_AGG / "best_val_median_ensemble.csv")
save_csv(bme_df, OUT_TABLES / "best_val_median_ensemble.csv")

display(bme_df)

# Copy summary figures to global figures.
copy_if_exists(ENS_OUT / "hist_ensemble_pred_vs_true.png", OUT_FIGURES / "hist_ensemble_pred_vs_true.png")
copy_if_exists(ENS_OUT / "scatter_ensemble_pred_vs_true.png", OUT_FIGURES / "scatter_ensemble_pred_vs_true.png")


In [ ]:
# =========================
# Loss diagnostics and overfit/underfit analysis
# =========================
section("Cell | Loss diagnostics and overfit/underfit analysis")

summary = pd.read_csv(OUT_FINAL_AGG / "final_runs_summary.csv")
gap_rows = []
all_histories = []
for _, row in summary.iterrows():
    run_id = str(row["run_id"])
    hpath = OUT_FINAL_RUNS / run_id / "history.csv"
    if not hpath.exists():
        continue
    h = pd.read_csv(hpath).sort_values("epoch").reset_index(drop=True)
    # history.csv dari training menyimpan val_mae_mean.
    # Untuk artefak post-hoc yang kompatibel dengan WavLM,
    # val_mae_mean dipakai sebagai alias val_loss.
    if "val_loss" not in h.columns:
        if "val_mae_mean" not in h.columns:
            raise KeyError(f"history.csv {hpath} tidak punya kolom val_mae_mean/val_loss")
        h["val_loss"] = h["val_mae_mean"]
    
    if "train_val_gap" not in h.columns:
        h["train_val_gap"] = h["val_loss"] - h["train_loss"]
    
    h["run_id"] = run_id
    all_histories.append(h)
    best_epoch = int(row["best_epoch"])
    best_h = h.loc[h["epoch"] == best_epoch].iloc[0] if (h["epoch"] == best_epoch).any() else h.sort_values("val_S", ascending=False).iloc[0]
    last_h = h.iloc[-1]
    gap_best = float(best_h["val_mae_mean"] - best_h["train_loss"])
    gap_last = float(last_h["val_mae_mean"] - last_h["train_loss"])
    if gap_best <= 0.01 and gap_last <= 0.015:
        diagnosis = "tidak menunjukkan overfitting berat"
    elif gap_best <= 0.02 or gap_last <= 0.03:
        diagnosis = "indikasi gap ringan; perlu dibaca bersama kurva"
    else:
        diagnosis = "indikasi overfitting yang perlu dicermati"
    if len(h) >= 2:
        train_improve = float(h["train_loss"].iloc[0] - h["train_loss"].iloc[-1])
        val_improve = float(h["val_mae_mean"].iloc[0] - h["val_mae_mean"].min())
    else:
        train_improve = np.nan
        val_improve = np.nan
    if np.isfinite(train_improve) and np.isfinite(val_improve) and train_improve < 0.001 and val_improve < 0.001:
        underfit_note = "kemungkinan underfitting/optimisasi belum efektif"
    else:
        underfit_note = "tidak ada indikasi underfitting yang jelas dari loss"
    gap_rows.append({
    "run_id": run_id,
    "seed": int(row["seed"]),
    "best_epoch": best_epoch,
    "stop_epoch": int(row["stop_epoch"]),
    "final_epoch": int(h["epoch"].max()),

    "min_train_loss": float(h["train_loss"].min()),
    "min_val_loss": float(h["val_loss"].min()),
    "epoch_min_val_loss": int(h.loc[h["val_loss"].idxmin(), "epoch"]),

    "train_loss_at_best_epoch": float(best_h["train_loss"]),
    "val_loss_at_best_epoch": float(best_h["val_loss"]),
    "gap_at_best_epoch": float(best_h["train_val_gap"]),

    "final_train_loss": float(last_h["train_loss"]),
    "final_val_loss": float(last_h["val_loss"]),
    "final_gap": float(last_h["train_val_gap"]),

    "mean_gap": float(h["train_val_gap"].mean()),
    "max_gap": float(h["train_val_gap"].max()),
    "min_gap": float(h["train_val_gap"].min()),

    "diagnosis": diagnosis,

    # legacy HuBERT columns, boleh tetap disimpan
    "overfit_diagnosis": diagnosis,
    "underfit_note": underfit_note,
})

gap_df = pd.DataFrame(gap_rows)
save_csv(gap_df, OUT_FINAL_AGG / "train_val_gap_diagnostics.csv")
save_csv(gap_df, OUT_TABLES / "train_val_gap_diagnostics.csv")
# WavLM posthoc-compatible alias.
save_csv(gap_df, OUT_FINAL_AGG / "loss_diagnostic_summary_per_run.csv")
save_csv(gap_df, OUT_TABLES / "loss_diagnostic_summary_per_run.csv")
save_csv(gap_df[["run_id", "overfit_diagnosis", "underfit_note"]], OUT_TABLES / "overfit_underfit_diagnosis.csv")
display(gap_df)

# Per-run posthoc-compatible history files
history_index_rows = []

for _, row in summary.iterrows():
    run_id = str(row["run_id"])
    run_dir = OUT_FINAL_RUNS / run_id
    hpath = run_dir / "history.csv"

    history_index_rows.append({
        "run_id": run_id,
        "seed": int(row["seed"]),
        "run_dir": str(run_dir),
        "history_path": str(hpath),
        "history_exists": hpath.exists(),
    })

    if hpath.exists():
        h = pd.read_csv(hpath).copy()
        h["val_loss"] = h["val_mae_mean"]
        h["train_val_gap"] = h["val_loss"] - h["train_loss"]
        save_csv(h, run_dir / "history_with_val_loss.csv")

save_csv(pd.DataFrame(history_index_rows), OUT_FINAL_AGG / "final_run_history_index.csv")
save_csv(pd.DataFrame(history_index_rows), OUT_TABLES / "final_run_history_index.csv")


if all_histories:
    hist_all = pd.concat(all_histories, ignore_index=True)
    hist_all["val_loss"] = hist_all["val_mae_mean"]
    hist_all["train_val_gap"] = hist_all["val_loss"] - hist_all["train_loss"]
    save_csv(hist_all, OUT_FINAL_AGG / "all_final_histories_long.csv")
    # WavLM posthoc-compatible alias.
    save_csv(hist_all, OUT_FINAL_AGG / "all_runs_history_with_val_loss.csv")
    agg = hist_all.groupby("epoch").agg(
    train_loss_mean=("train_loss", "mean"),
    train_loss_std=("train_loss", "std"),
    val_loss_mean=("val_loss", "mean"),
    val_loss_std=("val_loss", "std"),
    train_val_gap_mean=("train_val_gap", "mean"),
    train_val_gap_std=("train_val_gap", "std"),
    n=("run_id", "nunique"),
    ).reset_index()
    
    
    # WavLM posthoc-compatible aliases
    agg["n_runs"] = agg["n"]
    agg["gap_mean"] = agg["train_val_gap_mean"]
    agg["gap_std"] = agg["train_val_gap_std"]

    # Optional: reorder front columns for posthoc compatibility
    front_cols = [
        "epoch",
        "n_runs",
        "train_loss_mean",
        "train_loss_std",
        "val_loss_mean",
        "val_loss_std",
        "gap_mean",
        "gap_std",
    ]
    remaining_cols = [c for c in agg.columns if c not in front_cols]
    agg = agg[front_cols + remaining_cols]
        
    save_csv(agg, OUT_FINAL_AGG / "aggregate_loss_by_epoch.csv")
    # WavLM posthoc-compatible alias.
    save_csv(agg, OUT_FINAL_AGG / "aggregate_loss_curve_by_epoch.csv")
    save_csv(agg, OUT_TABLES / "aggregate_loss_curve_by_epoch.csv")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(agg["epoch"], agg["train_loss_mean"], marker="o", label="train_loss mean")
    ax.plot(agg["epoch"], agg["val_loss_mean"], marker="o", label="validation_loss / val_MAE_mean")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MAE / L1 loss")
    ax.set_title("Aggregate Train and Validation Loss Across Final Runs")
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_and_maybe_show(fig, OUT_FINAL_AGG / "aggregate_loss_curve.png", show=DISPLAY_SUMMARY_PLOTS)
    copy_if_exists(OUT_FINAL_AGG / "aggregate_loss_curve.png", OUT_FIGURES / "aggregate_loss_curve.png")
    
    
    # WavLM posthoc-compatible overlay plot: all runs train vs validation loss
    fig, ax = plt.subplots(figsize=(10, 6))
    for run_id, sub in hist_all.groupby("run_id"):
        ax.plot(sub["epoch"], sub["train_loss"], alpha=0.35)
        ax.plot(sub["epoch"], sub["val_loss"], alpha=0.35, linestyle="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss / MAE")
    ax.set_title("All Runs Train vs Validation Loss Overlay")
    ax.grid(True, alpha=0.3)
    save_and_maybe_show(
        fig,
        OUT_FINAL_AGG / "all_runs_train_vs_val_loss_overlay.png",
        show=DISPLAY_SUMMARY_PLOTS,
    )
    copy_if_exists(
        OUT_FINAL_AGG / "all_runs_train_vs_val_loss_overlay.png",
        OUT_FIGURES / "all_runs_train_vs_val_loss_overlay.png",
    )

    # WavLM posthoc-compatible mean ± std plot
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(agg["epoch"], agg["train_loss_mean"], label="train_loss mean")
    ax.plot(agg["epoch"], agg["val_loss_mean"], label="val_loss mean")
    ax.fill_between(
        agg["epoch"],
        agg["train_loss_mean"] - agg["train_loss_std"].fillna(0),
        agg["train_loss_mean"] + agg["train_loss_std"].fillna(0),
        alpha=0.15,
    )
    ax.fill_between(
        agg["epoch"],
        agg["val_loss_mean"] - agg["val_loss_std"].fillna(0),
        agg["val_loss_mean"] + agg["val_loss_std"].fillna(0),
        alpha=0.15,
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss / MAE")
    ax.set_title("Aggregate Train vs Validation Loss Mean ± Std")
    save_and_maybe_show(
        fig,
        OUT_FINAL_AGG / "aggregate_train_vs_val_loss_mean_std.png",
        show=DISPLAY_SUMMARY_PLOTS,
    )
    copy_if_exists(
        OUT_FINAL_AGG / "aggregate_train_vs_val_loss_mean_std.png",
        OUT_FIGURES / "aggregate_train_vs_val_loss_mean_std.png",
    )

    # WavLM posthoc-compatible train-validation gap plot
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(agg["epoch"], agg["train_val_gap_mean"], label="gap mean")
    ax.fill_between(
        agg["epoch"],
        agg["train_val_gap_mean"] - agg["train_val_gap_std"].fillna(0),
        agg["train_val_gap_mean"] + agg["train_val_gap_std"].fillna(0),
        alpha=0.15,
    )
    ax.axhline(0, linestyle="--")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss - Train loss")
    ax.set_title("Aggregate Train-Validation Gap Mean ± Std")
    save_and_maybe_show(
        fig,
        OUT_FINAL_AGG / "aggregate_train_val_gap_mean_std.png",
        show=DISPLAY_SUMMARY_PLOTS,
    )
    copy_if_exists(
        OUT_FINAL_AGG / "aggregate_train_val_gap_mean_std.png",
        OUT_FIGURES / "aggregate_train_val_gap_mean_std.png",
    )


if not gap_df.empty:
    overfit_count = gap_df["overfit_diagnosis"].str.contains(
        "overfitting", case=False, na=False
    ).sum()

    underfit_count = gap_df["underfit_note"].str.contains(
        "underfitting", case=False, na=False
    ).sum()

    interpretation_text = f"""
Interpretasi Diagnostik Train Loss dan Validation Loss

Analisis ini membaca history.csv dari setiap final run HuBERT + LoRA pada strict split.
Train loss dihitung sebagai MAE/L1 loss, sedangkan validation loss direpresentasikan oleh val_mae_mean.
Jumlah run terbaca: {len(gap_df)}.
Jumlah run dengan indikasi overfitting yang perlu dicermati: {overfit_count}.
Jumlah run dengan indikasi underfitting/optimisasi belum efektif: {underfit_count}.

Interpretasi ini bersifat post-hoc dan tidak mengubah hasil eksperimen utama.
""".strip()

    save_text(
        interpretation_text,
        OUT_FINAL_AGG / "overfit_underfit_interpretation.txt",
    )
    copy_if_exists(
        OUT_FINAL_AGG / "overfit_underfit_interpretation.txt",
        OUT_APPENDIX / "appendix_overfit_underfit_interpretation.txt",
    )

In [ ]:
# =========================
# R² / regression-to-the-mean diagnostics and per-trait artifacts
# =========================
section("Cell | R2 and regression-to-the-mean diagnostics")

BEST_VAL_PRED = OUT_FINAL_SELECTED / "best_val" / "predictions_test_strict.csv"
MEDIAN_PRED = OUT_FINAL_SELECTED / "median" / "predictions_test_strict.csv"
ENS_PRED = OUT_FINAL_SELECTED / "ensemble" / "ensemble_predictions_test_strict.csv"
assert BEST_VAL_PRED.exists(), f"Missing: {BEST_VAL_PRED}"

def save_trait_outputs(pred_path: Path, tag: str):
    dfp = pd.read_csv(pred_path)
    y_true = dfp[[f"true_{c}" for c in LABEL_COLS]].values
    y_pred = dfp[[f"pred_{c}" for c in LABEL_COLS]].values
    trait_df = per_trait_metrics_df(y_true, y_pred, tag=tag)
    save_csv(trait_df, OUT_TABLES / f"per_trait_metrics_{tag}.csv")
    plot_bar_metrics(trait_df, "MAE", OUT_FIGURES / f"per_trait_mae_{tag}.png", f"Per-trait MAE ({tag})", show=DISPLAY_SUMMARY_PLOTS)
    plot_bar_metrics(trait_df, "R2", OUT_FIGURES / f"per_trait_r2_{tag}.png", f"Per-trait R² ({tag})", show=DISPLAY_SUMMARY_PLOTS)
    return dfp, trait_df

best_df, best_trait_df = save_trait_outputs(BEST_VAL_PRED, "best_val")
save_csv(best_trait_df, OUT_FINAL_AGG / "best_val_test_trait_summary.csv")
save_csv(best_trait_df, OUT_TABLES / "best_val_test_trait_summary.csv")
if "mae_per_sample" not in best_df.columns:
    best_df["mae_per_sample"] = best_df[[f"abs_err_{c}" for c in LABEL_COLS]].mean(axis=1)

df_worst = best_df.sort_values("mae_per_sample", ascending=False).head(20).reset_index(drop=True)
df_best = best_df.sort_values("mae_per_sample", ascending=True).head(20).reset_index(drop=True)

save_csv(df_worst, OUT_FINAL_AGG / "top20_worst_predictions.csv")
save_csv(df_best, OUT_FINAL_AGG / "top20_best_predictions.csv")
save_csv(df_worst, OUT_TABLES / "top20_worst_predictions.csv")
save_csv(df_best, OUT_TABLES / "top20_best_predictions.csv")

if MEDIAN_PRED.exists():
    median_df, median_trait_df = save_trait_outputs(MEDIAN_PRED, "median")
if ENS_PRED.exists():
    ens_df, ens_trait_df = save_trait_outputs(ENS_PRED, "ensemble")

# Regression-to-the-mean summary for best-val run.
analysis_rows = []
for col in LABEL_COLS:
    true_vals = best_df[f"true_{col}"].values
    pred_vals = best_df[f"pred_{col}"].values
    true_var = float(np.var(true_vals))
    pred_var = float(np.var(pred_vals))
    compression = float(pred_var / true_var) if true_var > 1e-12 else np.nan
    corr = float(np.corrcoef(true_vals, pred_vals)[0, 1]) if len(true_vals) > 1 else np.nan
    analysis_rows.append({
        "trait": col,
        "true_mean": float(np.mean(true_vals)),
        "true_std": float(np.std(true_vals)),
        "true_var": true_var,
        "true_min": float(np.min(true_vals)),
        "true_max": float(np.max(true_vals)),
        "pred_mean": float(np.mean(pred_vals)),
        "pred_std": float(np.std(pred_vals)),
        "pred_var": pred_var,
        "pred_min": float(np.min(pred_vals)),
        "pred_max": float(np.max(pred_vals)),
        "pred_var_over_true_var": compression,
        "corr_true_pred": corr,
        "mae": float(np.mean(np.abs(pred_vals - true_vals))),
        "rmse": float(np.sqrt(np.mean((pred_vals - true_vals) ** 2))),
        "r2": float(r2_score(true_vals, pred_vals)),
    })
analysis_df = pd.DataFrame(analysis_rows)
save_csv(analysis_df, OUT_FINAL_AGG / "r2_vs_mae_analysis.csv")
save_csv(analysis_df, OUT_FINAL_AGG / "regression_to_mean_summary.csv")
save_csv(analysis_df, OUT_TABLES / "regression_to_mean_summary.csv")
display(analysis_df)

# Std comparison plot.
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(analysis_df))
width = 0.35
ax.bar(x - width/2, analysis_df["true_std"], width, label="true_std")
ax.bar(x + width/2, analysis_df["pred_std"], width, label="pred_std")
ax.set_xticks(x)
ax.set_xticklabels(analysis_df["trait"], rotation=30)
ax.set_ylabel("Standard deviation")
ax.set_title("True vs Predicted Standard Deviation (Best-Val Run)")
ax.legend()
save_and_maybe_show(fig, OUT_FIGURES / "true_std_vs_pred_std_best_val.png", show=DISPLAY_SUMMARY_PLOTS)

# Bin analysis.
bin_rows = []
for col in LABEL_COLS:
    temp = best_df[[f"true_{col}", f"pred_{col}"]].copy()
    temp.columns = ["true", "pred"]
    temp["bin"] = pd.qcut(temp["true"], q=5, duplicates="drop")
    grp = temp.groupby("bin", observed=False).agg(true_mean=("true", "mean"), pred_mean=("pred", "mean"), count=("true", "size")).reset_index()
    grp["trait"] = col
    grp["gap_pred_minus_true"] = grp["pred_mean"] - grp["true_mean"]
    bin_rows.append(grp)
bin_df = pd.concat(bin_rows, ignore_index=True)
save_csv(bin_df, OUT_FINAL_AGG / "best_val_test_bin_analysis.csv")
save_csv(bin_df, OUT_TABLES / "best_val_test_bin_analysis.csv")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(LABEL_COLS):
    ax = axes[i]
    sub = bin_df[bin_df["trait"] == col].reset_index(drop=True)
    ax.plot(range(len(sub)), sub["true_mean"], marker="o", label="true_mean")
    ax.plot(range(len(sub)), sub["pred_mean"], marker="o", label="pred_mean")
    ax.set_title(col)
    ax.set_xlabel("Target bin")
    ax.set_ylabel("Mean score")
    ax.legend()
axes[-1].axis("off")
save_and_maybe_show(fig, OUT_FIGURES / "best_val_test_bin_analysis.png", show=DISPLAY_SUMMARY_PLOTS)

# Residual boxplot.
residual_long = []
for col in LABEL_COLS:
    residual = best_df[f"pred_{col}"].values - best_df[f"true_{col}"].values
    for v in residual:
        residual_long.append({"trait": col, "residual": float(v)})
residual_df = pd.DataFrame(residual_long)
save_csv(residual_df, OUT_FINAL_AGG / "best_val_test_residuals_long.csv")
fig, ax = plt.subplots(figsize=(10, 5))
data = [residual_df[residual_df["trait"] == col]["residual"].values for col in LABEL_COLS]
ax.boxplot(data, tick_labels=LABEL_COLS)
ax.axhline(0, linestyle="--")
ax.set_title("Residual Distribution per Trait (Best-Val Test Strict)")
ax.set_ylabel("Pred - True")
save_and_maybe_show(fig, OUT_FIGURES / "best_val_test_residual_boxplot.png", show=DISPLAY_SUMMARY_PLOTS)

copy_if_exists(OUT_FIGURES / "true_std_vs_pred_std_best_val.png", OUT_FIGURES / "true_std_vs_pred_std.png")
copy_if_exists(OUT_FIGURES / "per_trait_mae_best_val.png", OUT_FIGURES / "per_trait_mae.png")
copy_if_exists(OUT_FIGURES / "per_trait_r2_best_val.png", OUT_FIGURES / "per_trait_r2.png")
copy_if_exists(OUT_FINAL_SELECTED / "best_val" / "hist_test_strict_pred_vs_true.png", OUT_FIGURES / "histogram_true_vs_pred_best_val.png")
copy_if_exists(OUT_FINAL_SELECTED / "best_val" / "scatter_test_strict_pred_vs_true.png", OUT_FIGURES / "scatter_true_vs_pred_best_val.png")


In [ ]:
# =========================
# WavLM-compatible posthoc diagnostics aliases
# =========================
section("Cell | Posthoc diagnostics aliases")

OUT_POSTHOC = OUTPUTS / "posthoc_diagnostics"
OUT_POSTHOC.mkdir(parents=True, exist_ok=True)

posthoc_alias_files = [
    # Loss diagnostics / overfit-underfit artifacts
    OUT_FINAL_AGG / "loss_diagnostic_summary_per_run.csv",
    OUT_FINAL_AGG / "train_val_gap_diagnostics.csv",
    OUT_FINAL_AGG / "all_final_histories_long.csv",
    OUT_FINAL_AGG / "all_runs_history_with_val_loss.csv",
    OUT_FINAL_AGG / "aggregate_loss_by_epoch.csv",
    OUT_FINAL_AGG / "aggregate_loss_curve_by_epoch.csv",
    OUT_FINAL_AGG / "aggregate_loss_curve.png",
    OUT_FINAL_AGG / "aggregate_train_vs_val_loss_mean_std.png",
    OUT_FINAL_AGG / "aggregate_train_val_gap_mean_std.png",
    OUT_FINAL_AGG / "all_runs_train_vs_val_loss_overlay.png",
    OUT_FINAL_AGG / "overfit_underfit_interpretation.txt",

    # R2 / regression-to-the-mean artifacts
    OUT_FINAL_AGG / "regression_to_mean_summary.csv",
    OUT_FINAL_AGG / "best_val_test_trait_summary.csv",
    OUT_FINAL_AGG / "best_val_test_bin_analysis.csv",
    OUT_FINAL_AGG / "best_val_test_residuals_long.csv",
    OUT_FINAL_AGG / "top20_worst_predictions.csv",
    OUT_FINAL_AGG / "top20_best_predictions.csv",

    # Figures
    OUT_FIGURES / "true_std_vs_pred_std_best_val.png",
    OUT_FIGURES / "true_std_vs_pred_std.png",
    OUT_FIGURES / "per_trait_mae_best_val.png",
    OUT_FIGURES / "per_trait_mae.png",
    OUT_FIGURES / "per_trait_r2_best_val.png",
    OUT_FIGURES / "per_trait_r2.png",
    OUT_FIGURES / "best_val_test_bin_analysis.png",
    OUT_FIGURES / "best_val_test_residual_boxplot.png",
    OUT_FIGURES / "histogram_true_vs_pred_best_val.png",
    OUT_FIGURES / "scatter_true_vs_pred_best_val.png",
]

copied_rows = []
for src in posthoc_alias_files:
    dst = OUT_POSTHOC / src.name
    copied = copy_if_exists(src, dst)
    copied_rows.append({
        "source": str(src.relative_to(ROOT)) if src.exists() else str(src),
        "alias_path": str(dst.relative_to(ROOT)),
        "source_exists": src.exists(),
        "copied": copied,
    })

posthoc_alias_df = pd.DataFrame(copied_rows)
save_csv(posthoc_alias_df, OUT_POSTHOC / "posthoc_alias_index.csv")
save_csv(posthoc_alias_df, OUT_TABLES / "posthoc_alias_index.csv")

display(posthoc_alias_df)
log_line(f"Posthoc alias folder prepared: {OUT_POSTHOC}", LIVE_LOG)

In [ ]:
# =========================
# Appendix-ready exports and artifact index
# =========================
section("Cell | Appendix-ready exports and artifact index")

# Copy selected prediction files.
copy_if_exists(OUT_FINAL_SELECTED / "best_val" / "predictions_test_strict.csv", OUT_APPENDIX / "appendix_best_val_predictions_test_strict.csv")
copy_if_exists(OUT_FINAL_SELECTED / "median" / "predictions_test_strict.csv", OUT_APPENDIX / "appendix_median_predictions_test_strict.csv")
copy_if_exists(OUT_FINAL_SELECTED / "ensemble" / "ensemble_predictions_test_strict.csv", OUT_APPENDIX / "appendix_ensemble_predictions_test_strict.csv")

# Copy core tables.
for src in [
    OUT_FINAL_AGG / "final_runs_summary.csv",
    OUT_FINAL_AGG / "final_stats_summary.csv",
    OUT_FINAL_AGG / "selected_runs_summary.json",
    OUT_FINAL_AGG / "train_val_gap_diagnostics.csv",
    OUT_FINAL_AGG / "r2_vs_mae_analysis.csv",
    OUT_OPTUNA_STUDY / "best_hp.csv",
    OUT_OPTUNA_STUDY / "optuna_top_trials.csv",
]:
    if src.exists():
        copy_if_exists(src, OUT_APPENDIX / f"appendix_{src.name}")

# Copy important figures.
important_figs = [
    OUT_OPTUNA_STUDY / "optuna_optimization_history.png",
    OUT_OPTUNA_STUDY / "optuna_param_importance.png",
    OUT_FINAL_AGG / "aggregate_loss_curve.png",
    OUT_FIGURES / "per_trait_mae_best_val.png",
    OUT_FIGURES / "per_trait_r2_best_val.png",
    OUT_FIGURES / "true_std_vs_pred_std_best_val.png",
    OUT_FIGURES / "best_val_test_bin_analysis.png",
    OUT_FIGURES / "best_val_test_residual_boxplot.png",
    OUT_FINAL_SELECTED / "best_val" / "hist_test_strict_pred_vs_true.png",
    OUT_FINAL_SELECTED / "best_val" / "scatter_test_strict_pred_vs_true.png",
    OUT_FINAL_SELECTED / "ensemble" / "hist_ensemble_pred_vs_true.png",
    OUT_FINAL_SELECTED / "ensemble" / "scatter_ensemble_pred_vs_true.png",
]
for src in important_figs:
    if src.exists():
        copy_if_exists(src, OUT_APPENDIX / src.name)

optuna_note = f"""
Optuna reproducibility note:
- N_TRIALS = {N_TRIALS}
- Remaining trials dihitung dari total len(study.trials), bukan jumlah COMPLETE.
- Sampler tidak dibuat eksplisit agar sesuai dengan notebook WavLM.
- Pruner = {PRUNER}
- Selection metric = validation S = 1 - MAE mean.
""".strip()

save_text(optuna_note, OUT_FINAL_AGG / "optuna_reproducibility_note.txt")
copy_if_exists(OUT_FINAL_AGG / "optuna_reproducibility_note.txt", OUT_APPENDIX / "appendix_optuna_reproducibility_note.txt")


strict_audit_csv = OUT_TABLES / "strict_split_leakage_audit.csv"
if strict_audit_csv.exists():
    audit_text = pd.read_csv(strict_audit_csv).to_string(index=False)
    save_text(audit_text, OUT_APPENDIX / "appendix_strict_split_leakage_audit.txt")


# Artifact index.
artifact_rows = []
for root_dir in [OUTPUTS, CHECKPOINTS, LOGS]:
    for p in root_dir.rglob("*"):
        if p.is_file():
            artifact_rows.append({
                "artifact": str(p.relative_to(ROOT)),
                "size_mb": round(p.stat().st_size / (1024**2), 4),
                "suffix": p.suffix,
            })
artifact_df = pd.DataFrame(artifact_rows).sort_values("artifact").reset_index(drop=True)
save_csv(artifact_df, OUT_APPENDIX / "artifact_index.csv")
display(artifact_df.tail(30))

log_line("Notebook completed successfully.", LIVE_LOG)
log_line(f"Appendix folder: {OUT_APPENDIX}", LIVE_LOG)



